In [2]:
# 您好！这是在您本地 Jupyter 环境中运行的完整代码。
# 请确保您的原始 Excel 文件 "ASI分类.xls" 与您的 Jupyter 笔记本在同一个文件夹中。

# 步骤 0: 安装所需的库 (如果您还没装)
# !pip install pandas scikit-learn openpyxl

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings

# 忽略将来可能的警告
warnings.filterwarnings('ignore', category=FutureWarning)

print("--- 机器学习数据预处理流程开始 (Jupyter 本地版) ---")

try:
    # 1. & 2. 加载原始 Excel 文件并进行初次清理
    print("步骤 1/9: 加载原始 Excel 文件 'ASI分类.xls' (Sheet1)...")
    
    # 【*** 这是关键修改 ***】
    # 我们直接读取 Excel (.xls) 文件，而不是 CSV
    # header=1 表示数据从第2行开始（第1行是表头）
    # 需要 'openpyxl' 库的支持
    df = pd.read_excel("ASI分类.xls", sheet_name="Sheet1", header=1) 
    # 【*** 修改结束 ***】

    print("步骤 2/9: 正在清理数据（将文本列转换为数字）...")
    # 找到 'SiO2' 和 'Cs ' 的索引位置
    try:
        start_col_index = df.columns.get_loc('SiO2')
        # 'Cs ' 列名在 Excel 读取时可能去掉了空格
        end_col_col = 'Cs ' if 'Cs ' in df.columns else 'Cs'
        
        if end_col_col not in df.columns:
             print(f"警告: 找不到 '{end_col_col}' 列, 转换到 'U' 列为止")
             end_col_index = df.columns.get_loc('U')
        else:
             end_col_index = df.columns.get_loc(end_col_col)

        cols_to_convert = df.columns[start_col_index : end_col_index + 1]
        
        converted_count = 0
        for col in cols_to_convert:
            if df[col].dtype == 'object':
                df[col] = pd.to_numeric(df[col], errors='coerce')
                converted_count += 1
        print(f"数据类型清理完毕。转换了 {converted_count} 个 'object' 列为 'numeric'。")

    except KeyError as e:
        print(f"警告: 找不到关键列 ({e})。跳过自动类型转换。")

    print(f"数据加载和清理完毕。 初始形状: {df.shape}")

    # 3. 定义特征 (X) 和目标 (y)
    print("步骤 3/9: 定义特征 (X) 和目标 (y)...")
    # 'Type-1' (目标泄露) 和 'No.' (索引) 必须被移除
    cols_to_drop_initial = ['No.', 'Type', 'Type-1']
    # 检查哪些列实际存在于DataFrame中，以避免KeyError
    cols_present = [col for col in cols_to_drop_initial if col in df.columns]
    X = df.drop(columns=cols_present)
    
    if 'Type' in df.columns:
        y = df['Type']
    else:
        raise ValueError("目标列 'Type' 在数据中未找到。")
    
    print(f"X (特征) 初始形状: {X.shape}")

    # 4. 剔除高缺失率特征
    print("步骤 4/9: 正在剔除缺失率 > 40% 的列...")
    missing_percentage = X.isnull().mean()
    cols_to_drop_missing = missing_percentage[missing_percentage > 0.4].index
    
    X = X.drop(columns=cols_to_drop_missing)
    
    if len(cols_to_drop_missing) > 0:
        print(f"已删除 {len(cols_to_drop_missing)} 个高缺失列: {list(cols_to_drop_missing)}")
    else:
        print("没有列的缺失率超过 40%。")
    print(f"X (特征) 剔除后形状: {X.shape}")
    
    # 5. 编码目标变量 (y)
    print("步骤 5/9: 正在编码目标变量 'Type' (A=0, I=1, S=2)...")
    le = LabelEncoder()
    # 确保 y 中没有 NaN 值
    y_clean = y.dropna()
    y_indices = y_clean.index
    # 确保 X 和 y 保持同步
    X_clean = X.loc[y_indices]
    
    y_encoded = le.fit_transform(y_clean)
    print(f"目标 'Type' 已被编码: {list(le.classes_)} -> {list(range(len(le.classes_)))}")

    # 6. 拆分数据为训练集和测试集
    print("步骤 6/9: 正在拆分训练集和测试集 (80/20 分层抽样)...")
    X_train, X_test, y_train, y_test = train_test_split(
        X_clean, y_encoded, 
        test_size=0.2, 
        random_state=42, 
        stratify=y_encoded # 确保 A/I/S-type 在训练集和测试集中比例相同
    )
    print(f"训练集形状: X_train: {X_train.shape}, y_train: {y_train.shape}")
    print(f"测试集形状: X_test: {X_test.shape}, y_test: {y_test.shape}")
    
    feature_names = X_clean.columns

    # 7. 插补缺失值 (KNNImputer)
    print(f"步骤 7/9: 正在使用 KNNImputer (k=5) 智能插补缺失值...")
    imputer = KNNImputer(n_neighbors=5)
    X_train_imputed = imputer.fit_transform(X_train)
    X_test_imputed = imputer.transform(X_test)
    print("插补完成。")

    # 8. 特征标准化 (StandardScaler)
    print("步骤 8/9: 正在使用 StandardScaler 标准化所有特征...")
    scaler = StandardScaler()
    X_train_processed = scaler.fit_transform(X_train_imputed)
    X_test_processed = scaler.transform(X_test_imputed)
    print("标准化完成。")

    # 9. 保存处理后的数据
    print("步骤 9/9: 正在保存处理后的数据到 CSV 文件...")
    X_train_df = pd.DataFrame(X_train_processed, columns=feature_names)
    y_train_df = pd.DataFrame(y_train, columns=['Type_Encoded'])
    train_processed_df = pd.concat([X_train_df, y_train_df], axis=1)
    
    X_test_df = pd.DataFrame(X_test_processed, columns=feature_names)
    y_test_df = pd.DataFrame(y_test, columns=['Type_Encoded'])
    test_processed_df = pd.concat([X_test_df, y_test_df], axis=1)
    
    train_csv_path = "train_processed.csv"
    test_csv_path = "test_processed.csv"
    
    train_processed_df.to_csv(train_csv_path, index=False)
    test_processed_df.to_csv(test_csv_path, index=False)
    
    print(f"\n--- 预处理流程全部完成 ---")
    print(f"已保存“机器学习就绪”的训练数据到: {train_csv_path}")
    print(f"已保存“机器学习就绪”的测试数据到: {test_csv_path}")

except FileNotFoundError:
    print(f"\n--- 错误: 文件未找到 ---")
    print(f"找不到文件 'ASI分类.xls'。")
    print("请确保您的 Excel 文件与 Jupyter 笔记本 (ipynb) 文件在同一个文件夹中。")
except ImportError:
    print(f"\n--- 错误: 缺少库 ---")
    print("请先安装 'openpyxl' 库来读取 Excel 文件。")
    print("您可以在 notebook 的一个单元格中运行: !pip install openpyxl")
except Exception as e:
    print(f"\n--- 处理过程中发生错误 ---")
    print(f"错误详情: {e}")

--- 机器学习数据预处理流程开始 (Jupyter 本地版) ---
步骤 1/9: 加载原始 Excel 文件 'ASI分类.xls' (Sheet1)...
步骤 2/9: 正在清理数据（将文本列转换为数字）...
数据类型清理完毕。转换了 32 个 'object' 列为 'numeric'。
数据加载和清理完毕。 初始形状: (1341, 50)
步骤 3/9: 定义特征 (X) 和目标 (y)...
X (特征) 初始形状: (1341, 47)
步骤 4/9: 正在剔除缺失率 > 40% 的列...
已删除 1 个高缺失列: ['Cs ']
X (特征) 剔除后形状: (1341, 46)
步骤 5/9: 正在编码目标变量 'Type' (A=0, I=1, S=2)...
目标 'Type' 已被编码: ['A-type', 'I-type', 'S-type'] -> [0, 1, 2]
步骤 6/9: 正在拆分训练集和测试集 (80/20 分层抽样)...
训练集形状: X_train: (1072, 46), y_train: (1072,)
测试集形状: X_test: (269, 46), y_test: (269,)
步骤 7/9: 正在使用 KNNImputer (k=5) 智能插补缺失值...
插补完成。
步骤 8/9: 正在使用 StandardScaler 标准化所有特征...
标准化完成。
步骤 9/9: 正在保存处理后的数据到 CSV 文件...

--- 预处理流程全部完成 ---
已保存“机器学习就绪”的训练数据到: train_processed.csv
已保存“机器学习就绪”的测试数据到: test_processed.csv


In [9]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import warnings

# 忽略将来可能的警告
warnings.filterwarnings('ignore', category=FutureWarning)

print("--- 机器学习模型训练与评估开始 ---")

try:
    # 1. 加载您刚刚生成的处理后数据
    print("步骤 1/5: 正在加载 'train_processed.csv' 和 'test_processed.csv'...")
    train_df = pd.read_csv("train_processed.csv")
    test_df = pd.read_csv("test_processed.csv")
    print("数据加载完毕。")

    # 2. 再次分离特征 (X) 和目标 (y)
    print("步骤 2/5: 正在分离特征 (X) 和目标 (y)...")
    
    # 目标列是我们之前编码的 'Type_Encoded'
    target_column = 'Type_Encoded'
    
    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]
    
    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]
    
    print(f"训练集: X_train {X_train.shape}, y_train {y_train.shape}")
    print(f"测试集: X_test {X_test.shape}, y_test {y_test.shape}")

    # 3. 初始化和训练模型（逻辑回归）
    print("步骤 3/5: 正在初始化逻辑回归模型...")
    # 'multi_class='ovr'' 表示 "One-vs-Rest"，适合处理三个或更多类别
    # 'max_iter=1000' 确保模型有足够的时间收敛
    model = LogisticRegression(multi_class='ovr', max_iter=1000, random_state=42)
    
    print("步骤 4/5: 正在使用训练数据 (X_train) 训练模型...")
    model.fit(X_train, y_train)
    print("模型训练完成！")

    # 5. 评估模型
    print("步骤 5/5: 正在使用测试数据 (X_test) 评估模型性能...")
    
    # (A) 在测试集上进行预测
    y_pred = model.predict(X_test)
    
    # (B) 计算总体准确率
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\n--- 模型评估结果 (逻辑回归) ---")
    print(f"\n[ 1. 总体准确率 ]\n模型在测试集上的准确率为: {accuracy * 100:.2f}%")
    
    # (C) 打印详细的分类报告
    # 别忘了 'Type' 的编码: 0='A-type', 1='I-type', 2='S-type'
    target_names = ['A-type (Class 0)', 'I-type (Class 1)', 'S-type (Class 2)']
    report = classification_report(y_test, y_pred, target_names=target_names)
    
    print(f"\n[ 2. 详细分类报告 ]\n{report}")
    print("--- 流程结束 ---")

except FileNotFoundError:
    print("\n--- 错误: 文件未找到 ---")
    print("找不到 'train_processed.csv' 或 'test_processed.csv'。")
    print("请确保此脚本与您上一步生成的文件在同一个文件夹中。")
except Exception as e:
    print(f"\n--- 处理过程中发生错误 ---")
    print(f"错误详情: {e}")

--- 机器学习模型训练与评估开始 ---
步骤 1/5: 正在加载 'train_processed.csv' 和 'test_processed.csv'...
数据加载完毕。
步骤 2/5: 正在分离特征 (X) 和目标 (y)...
训练集: X_train (1072, 46), y_train (1072,)
测试集: X_test (269, 46), y_test (269,)
步骤 3/5: 正在初始化逻辑回归模型...
步骤 4/5: 正在使用训练数据 (X_train) 训练模型...
模型训练完成！
步骤 5/5: 正在使用测试数据 (X_test) 评估模型性能...

--- 模型评估结果 (逻辑回归) ---

[ 1. 总体准确率 ]
模型在测试集上的准确率为: 86.25%

[ 2. 详细分类报告 ]
                  precision    recall  f1-score   support

A-type (Class 0)       0.90      0.94      0.92       155
I-type (Class 1)       0.82      0.79      0.81        81
S-type (Class 2)       0.79      0.67      0.72        33

        accuracy                           0.86       269
       macro avg       0.83      0.80      0.81       269
    weighted avg       0.86      0.86      0.86       269

--- 流程结束 ---


In [11]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
import numpy as np

# 忽略将来可能的警告
warnings.filterwarnings('ignore', category=FutureWarning)

print("--- 机器学习模型优化 (随机森林) ---")

try:
    # 1. 加载您上一步生成的处理后数据
    print("步骤 1/6: 正在加载 'train_processed.csv' 和 'test_processed.csv'...")
    train_df = pd.read_csv("train_processed.csv")
    test_df = pd.read_csv("test_processed.csv")
    print("数据加载完毕。")

    # 2. 再次分离特征 (X) 和目标 (y)
    print("步骤 2/6: 正在分离特征 (X) 和目标 (y)...")
    target_column = 'Type_Encoded'
    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]
    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]

    # 获取特征名称列表，以便后续显示“重要特征”
    feature_names = X_train.columns

    print(f"训练集: X_train {X_train.shape}, y_train {y_train.shape}")
    print(f"测试集: X_test {X_test.shape}, y_test {y_test.shape}")

    # 3. 初始化和训练模型（随机森林）
    print("步骤 3/6: 正在初始化随机森林分类器...")
    # n_estimators=100 表示我们将构建 100 棵决策树
    # random_state=42 确保结果可复现
    # n_jobs=-1 会使用您电脑的所有 CPU 核心来加速训练
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

    print("步骤 4/6: 正在使用训练数据 (X_train) 训练模型...")
    rf_model.fit(X_train, y_train)
    print("模型训练完成！")

    # 4. 评估模型
    print("步骤 5/6: 正在使用测试数据 (X_test) 评估模型性能...")
    # (A) 在测试集上进行预测
    y_pred_rf = rf_model.predict(X_test)

    # (B) 计算总体准确率
    accuracy_rf = accuracy_score(y_test, y_pred_rf)
    print(f"\n--- 模型评估结果 (随机森林) ---")
    print(f"\n[ 1. 总体准确率 ]")
    print(f"模型在测试集上的准确率为: {accuracy_rf * 100:.2f}%")
    print("(请与您之前的逻辑回归准确率 86.25% 进行对比)")

    # (C) 打印详细的分类报告
    target_names = ['A-type (Class 0)', 'I-type (Class 1)', 'S-type (Class 2)']
    report_rf = classification_report(y_test, y_pred_rf, target_names=target_names)
    print(f"\n[ 2. 详细分类报告 ]\n{report_rf}")
    print("请特别关注 S-type 的 Recall (召回率) 是否高于 0.67")

    # 5. 分析“特征重要性” (这是路线图中的第2点)
    print("\n步骤 6/6: 正在提取对分类最重要的特征...")
    importances = rf_model.feature_importances_

    # 将特征名称和它们的重要性得分打包在一起
    feature_importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    })

    # 按重要性从高到低排序
    feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

    print(f"\n[ 3. 特征重要性排名 (Top 15) ]")
    print("这告诉我们模型主要依赖哪些化学成分来进行分类：")
    # .to_string(index=False) 让输出更整洁
    print(feature_importance_df.head(15).to_string(index=False))

    print("\n--- 流程结束 ---")

except FileNotFoundError:
    print("\n--- 错误: 文件未找到 ---")
    print("找不到 'train_processed.csv' 或 'test_processed.csv'。")
    print("请确保此脚本与您上一步生成的文件在同一个文件夹中。")
except Exception as e:
    print(f"\n--- 处理过程中发生错误 ---")
    print(f"错误详情: {e}")

--- 机器学习模型优化 (随机森林) ---
步骤 1/6: 正在加载 'train_processed.csv' 和 'test_processed.csv'...
数据加载完毕。
步骤 2/6: 正在分离特征 (X) 和目标 (y)...
训练集: X_train (1072, 46), y_train (1072,)
测试集: X_test (269, 46), y_test (269,)
步骤 3/6: 正在初始化随机森林分类器...
步骤 4/6: 正在使用训练数据 (X_train) 训练模型...
模型训练完成！
步骤 5/6: 正在使用测试数据 (X_test) 评估模型性能...

--- 模型评估结果 (随机森林) ---

[ 1. 总体准确率 ]
模型在测试集上的准确率为: 89.59%
(请与您之前的逻辑回归准确率 86.25% 进行对比)

[ 2. 详细分类报告 ]
                  precision    recall  f1-score   support

A-type (Class 0)       0.92      0.95      0.93       155
I-type (Class 1)       0.85      0.83      0.84        81
S-type (Class 2)       0.90      0.82      0.86        33

        accuracy                           0.90       269
       macro avg       0.89      0.86      0.88       269
    weighted avg       0.90      0.90      0.90       269

请特别关注 S-type 的 Recall (召回率) 是否高于 0.67

步骤 6/6: 正在提取对分类最重要的特征...

[ 3. 特征重要性排名 (Top 15) ]
这告诉我们模型主要依赖哪些化学成分来进行分类：
    Feature  Importance
 Zr+Nb+Ce+Y    0.063173
         Hf    0.058097
 

In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE  # <-- 导入 SMOTE
import warnings
import numpy as np

# 忽略将来可能的警告
warnings.filterwarnings('ignore', category=FutureWarning)

print("--- 机器学习高级步骤 (SMOTE + 随机森林) ---")
print("目标：通过处理数据不平衡，看能否超越 89.59% 的准确率")

try:
    # 1. 加载处理后的数据
    print("\n步骤 1/6: 正在加载 'train_processed.csv' 和 'test_processed.csv'...")
    train_df = pd.read_csv("train_processed.csv")
    test_df = pd.read_csv("test_processed.csv")
    print("数据加载完毕。")

    # 2. 再次分离特征 (X) 和目标 (y)
    print("步骤 2/6: 正在分离特征 (X) 和目标 (y)...")
    target_column = 'Type_Encoded'
    
    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]
    
    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]
    
    # 打印 SMOTE 前的样本分布
    print("\n--- SMOTE 处理前 ---")
    print(f"原始训练集样本分布:\n{y_train.value_counts().sort_index()}")
    print("(0=A-type, 1=I-type, 2=S-type)")


    # 3. 【新步骤】应用 SMOTE 来平衡训练数据
    print("\n步骤 3/6: 正在应用 SMOTE 来平衡训练数据...")
    # SMOTE 会将少数类 (1 和 2) 的样本数量提升到与多数类 (0) 相同
    # n_jobs=-1 会使用所有CPU核心加速
    smote = SMOTE(random_state=42, n_jobs=-1)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    
    print("\n--- SMOTE 处理后 ---")
    print(f"新训练集样本分布:\n{y_train_smote.value_counts().sort_index()}")
    print("所有类别的样本数已变得相同。")


    # 4. 在 SMOTE 处理后的数据上训练模型
    print("\n步骤 4/6: 正在使用 (SMOTE) 平衡后的数据训练新模型...")
    rf_model_smote = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    
    # 使用新数据 (X_train_smote, y_train_smote) 进行训练
    rf_model_smote.fit(X_train_smote, y_train_smote)
    print("模型训练完成。")

    # 5. 在 *原始* 测试集上评估模型
    print("\n步骤 5/6: 正在使用 *原始* 测试集评估 (SMOTE) 模型...")
    # 评估必须在原始的、真实的 X_test 上进行
    y_pred_smote = rf_model_smote.predict(X_test)
    
    accuracy_smote = accuracy_score(y_test, y_pred_smote)
    print(f"\n--- 模型评估结果 (SMOTE + 随机森林) ---")
    print(f"\n[ 1. 总体准确率 ]")
    print(f"模型 (SMOTE) 准确率为: {accuracy_smote * 100:.2f}%")
    print("(对比：原随机森林 89.59% | 逻辑回归 86.25%)")
    
    # 6. 打印详细的分类报告
    print("\n[ 2. 详细分类报告 ]")
    target_names = ['A-type (Class 0)', 'I-type (Class 1)', 'S-type (Class 2)']
    report_smote = classification_report(y_test, y_pred_smote, target_names=target_names)
    print(report_smote)
    print("\n--- 流程结束 ---")

except FileNotFoundError:
    print("\n--- 错误: 文件未找到 ---")
    print("找不到 'train_processed.csv' 或 'test_processed.csv'。")
except ImportError:
    print(f"\n--- 错误: 缺少库 ---")
    print("请先安装 'imbalanced-learn' 库来使用 SMOTE。")
    print("您可以在 notebook 的一个单元格中运行: !pip install imbalanced-learn")
except Exception as e:
    print(f"\n--- 处理过程中发生错误 ---")
    print(f"错误详情: {e}")

--- 机器学习高级步骤 (SMOTE + 随机森林) ---
目标：通过处理数据不平衡，看能否超越 89.59% 的准确率

步骤 1/6: 正在加载 'train_processed.csv' 和 'test_processed.csv'...
数据加载完毕。
步骤 2/6: 正在分离特征 (X) 和目标 (y)...

--- SMOTE 处理前 ---
原始训练集样本分布:
0    619
1    324
2    129
Name: Type_Encoded, dtype: int64
(0=A-type, 1=I-type, 2=S-type)

步骤 3/6: 正在应用 SMOTE 来平衡训练数据...

--- SMOTE 处理后 ---
新训练集样本分布:
0    619
1    619
2    619
Name: Type_Encoded, dtype: int64
所有类别的样本数已变得相同。

步骤 4/6: 正在使用 (SMOTE) 平衡后的数据训练新模型...
模型训练完成。

步骤 5/6: 正在使用 *原始* 测试集评估 (SMOTE) 模型...

--- 模型评估结果 (SMOTE + 随机森林) ---

[ 1. 总体准确率 ]
模型 (SMOTE) 准确率为: 90.33%
(对比：原随机森林 89.59% | 逻辑回归 86.25%)

[ 2. 详细分类报告 ]
                  precision    recall  f1-score   support

A-type (Class 0)       0.93      0.94      0.93       155
I-type (Class 1)       0.88      0.83      0.85        81
S-type (Class 2)       0.84      0.94      0.89        33

        accuracy                           0.90       269
       macro avg       0.88      0.90      0.89       269
    weighted avg       0.90    

In [9]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import validation_curve # <-- 新增
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 忽略将来可能的警告
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 新增: 设置 Matplotlib 支持中文 ---
try:
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 尝试指定中文字体
    print("中文字体 'SimHei' 加载成功。")
except Exception as e:
    print(f"中文字体 'SimHei' 设置失败, 图表标题可能显示为方块: {e}")
    print("您可以尝试安装 'SimHei' 字体, 或在代码中更改为 'Microsoft YaHei' 等您已有的中文字体")

# 【*** 修改点 ***】
# 无论中文字体是否加载成功, 都 *必须* 设置这一项来修复负号显示问题
# 这一行命令告诉 matplotlib 使用标准的 ASCII 连字符 '-' 来显示负号
plt.rcParams['axes.unicode_minus'] = False
# 【*** 修改结束 ***】


print("--- 机器学习模型训练与评估开始 (逻辑回归) ---")


try:
    # 1. 加载您刚刚生成的处理后数据
    print("步骤 1/10: 正在加载 'train_processed.csv' 和 'test_processed.csv'...")
    train_df = pd.read_csv("train_processed.csv")
    test_df = pd.read_csv("test_processed.csv")
    print("数据加载完毕。")

    # 2. 再次分离特征 (X) 和目标 (y)
    print("步骤 2/10: 正在分离特征 (X) 和目标 (y)...")
    
    target_column = 'Type_Encoded'
    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]
    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]
    
    # 合并训练集和测试集 (用于绘图)
    X_all = pd.concat([X_train, X_test], axis=0)
    y_all = pd.concat([y_train, y_test], axis=0)
    
    # 保存特征名称以备后用
    feature_names = X_train.columns
    class_names_plot = ['A-type', 'I-type', 'S-type'] # 0, 1, 2
    
    print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")

    # 3. 初始化和训练模型（逻辑回归）
    print("步骤 3/10: 正在初始化逻辑回归模型...")
    model = LogisticRegression(multi_class='ovr', max_iter=1000, random_state=42)
    
    print("步骤 4/10: 正在使用训练数据 (X_train) 训练模型...")
    model.fit(X_train, y_train)
    print("模型训练完成！")

    # 5. 评估模型
    print("步骤 5/10: 正在使用测试数据 (X_test) 评估模型性能...")
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    print(f"\n--- 模型评估结果 (逻辑回归) ---")
    print(f"\n[ 1. 总体准确率 ]\n模型在测试集上的准确率为: {accuracy * 100:.2f}%")
    
    target_names_report = ['A-type (Class 0)', 'I-type (Class 1)', 'S-type (Class 2)']
    report = classification_report(y_test, y_pred, target_names=target_names_report)
    print(f"\n[ 2. 详细分类报告 ]\n{report}")
    
    # --- 6. 绘制混淆矩阵 ---
    print("\n步骤 6/10: 正在生成混淆矩阵图表...")
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names_plot, yticklabels=class_names_plot)
    plt.title(f'逻辑回归 混淆矩阵\n(总体准确率: {accuracy * 100:.2f}%)')
    plt.ylabel('真实类别 (True Label)')
    plt.xlabel('预测类别 (Predicted Label)')
    plot_filename_cm = "cm_logistic_regression.png"
    plt.tight_layout()
    plt.savefig(plot_filename_cm)
    print(f"图表已保存为: {plot_filename_cm}")
    plt.close()

    # --- 7. 绘制分类报告柱状图 ---
    print("\n步骤 7/10: 正在生成分类报告柱状图...")
    report_dict = classification_report(y_test, y_pred, target_names=class_names_plot, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    report_df = report_df.loc[class_names_plot, ['precision', 'recall', 'f1-score']]
    
    report_df.plot(kind='bar', figsize=(12, 7), rot=0)
    plt.title('逻辑回归 分类报告指标')
    plt.ylabel('得分 (Score)')
    plt.xlabel('岩石类别')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.ylim(0, 1.05)
    for p in plt.gca().patches:
        plt.gca().annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='center', xytext=(0, 9), textcoords='offset points')
    plot_filename_report = "report_bar_chart_lr.png"
    plt.tight_layout()
    plt.savefig(plot_filename_report)
    print(f"图表已保存为: {plot_filename_report}")
    plt.close()

    # --- 8. 绘制特征重要性柱状图 ---
    print("\n步骤 8/10: 正在生成特征重要性柱状图...")
    if hasattr(model, 'coef_'):
        avg_importance = np.mean(np.abs(model.coef_), axis=0)
        feature_importance_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': avg_importance
        }).sort_values(by='Importance', ascending=False)
        
        plt.figure(figsize=(12, 9))
        sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(15), palette='rocket')
        plt.title('逻辑回归 - Top 15 重要特征 (系数绝对值均值)')
        plt.xlabel('重要性得分 (Avg. Absolute Coefficient)')
        plt.ylabel('特征 (Feature)')
        
        plot_filename_features = "feature_importance_lr.png"
        plt.tight_layout()
        plt.savefig(plot_filename_features)
        print(f"图表已保存为: {plot_filename_features}")
        plt.close()
        
        # --- 9. 新增: 绘制特征散点图 ---
        print("\n步骤 9/10: 正在生成最重要特征散点图...")
        # 从上一步获取最重要的两个特征
        top_feature_1 = feature_importance_df.iloc[0]['Feature']
        top_feature_2 = feature_importance_df.iloc[1]['Feature']
        
        # 将 y (0,1,2) 映射回类别名称 (A, I, S)
        y_all_named = y_all.map({0: 'A-type', 1: 'I-type', 2: 'S-type'})
        
        # 创建包含绘图所需数据的 DataFrame
        scatter_df = pd.DataFrame({
            'Feature_1': X_all[top_feature_1],
            'Feature_2': X_all[top_feature_2],
            'Type': y_all_named
        })

        plt.figure(figsize=(10, 7))
        sns.scatterplot(data=scatter_df, x='Feature_1', y='Feature_2', hue='Type', alpha=0.7, palette='bright')
        plt.title(f'逻辑回归 最重要特征散点图\n({top_feature_1} vs {top_feature_2})')
        plt.xlabel(top_feature_1)
        plt.ylabel(top_feature_2)
        
        plot_filename_scatter = "scatter_top_features_lr.png"
        plt.tight_layout()
        plt.savefig(plot_filename_scatter)
        print(f"图表已保存为: {plot_filename_scatter}")
        plt.close()
        
    else:
        print("模型没有 'coef_' 属性，跳过特征重要性图和散点图。")

    # --- 10. 新增: 绘制验证曲线 (折线图) ---
    print("\n步骤 10/10: 正在生成验证曲线(折线图)... (这可能需要一点时间)")
    try:
        # 'C' 是逻辑回归的正则化参数，我们测试一系列值
        param_range = np.logspace(-3, 3, 7) # [0.001, 0.01, 0.1, 1, 10, 100, 1000]
        
        train_scores, test_scores = validation_curve(
            LogisticRegression(multi_class='ovr', max_iter=1000, random_state=42),
            X_all, y_all, # 使用所有数据来绘制曲线
            param_name="C", # 我们要调整的参数
            param_range=param_range,
            cv=3, # 3折交叉验证
            scoring="accuracy", # 评估指标
            n_jobs=1 # 【*** 修改点 ***】 改为 1 (串行计算), 避免内存崩溃
        )
        
        # 计算平均值和标准差
        train_scores_mean = np.mean(train_scores, axis=1)
        train_scores_std = np.std(train_scores, axis=1)
        test_scores_mean = np.mean(test_scores, axis=1)
        test_scores_std = np.std(test_scores, axis=1)

        plt.figure(figsize=(10, 6))
        plt.title("逻辑回归 验证曲线 (参数 'C')")
        plt.xlabel("正则化强度 C (值越大, 模型越复杂)")
        plt.ylabel("准确率 (Accuracy)")
        plt.ylim(0.0, 1.1)
        lw = 2
        
        # 设为对数刻度
        plt.semilogx(param_range, train_scores_mean, label="训练得分", color="darkorange", lw=lw)
        plt.fill_between(param_range, train_scores_mean - train_scores_std,
                         train_scores_mean + train_scores_std, alpha=0.2, color="darkorange")
        
        plt.semilogx(param_range, test_scores_mean, label="交叉验证得分 (测试)", color="navy", lw=lw)
        plt.fill_between(param_range, test_scores_mean - test_scores_std,
                         test_scores_mean + test_scores_std, alpha=0.2, color="navy")
        
        plt.legend(loc="best")
        
        plot_filename_validation = "validation_curve_lr.png"
        plt.tight_layout()
        plt.savefig(plot_filename_validation)
        print(f"图表已保存为: {plot_filename_validation}")
        plt.close()

    except Exception as e_val:
        print(f"验证曲线绘制失败: {e_val}")
    # --- 新增结束 ---

    print("\n--- 流程结束 (已生成最多5个图表) ---")

except FileNotFoundError:
    print("\n--- 错误: 文件未找到 ---")
    print("找不到 'train_processed.csv' 或 'test_processed.csv'。")
except ImportError:
    print(f"\n--- 错误: 缺少库 ---")
    print("请确保已安装 'matplotlib' 和 'seaborn' 库。")
except Exception as e:
    print(f"\n--- 处理过程中发生错误 ---")
    print(f"错误详情: {e}")




中文字体 'SimHei' 加载成功。
--- 机器学习模型训练与评估开始 (逻辑回归) ---
步骤 1/10: 正在加载 'train_processed.csv' 和 'test_processed.csv'...
数据加载完毕。
步骤 2/10: 正在分离特征 (X) 和目标 (y)...
训练集: (1072, 46), 测试集: (269, 46)
步骤 3/10: 正在初始化逻辑回归模型...
步骤 4/10: 正在使用训练数据 (X_train) 训练模型...
模型训练完成！
步骤 5/10: 正在使用测试数据 (X_test) 评估模型性能...

--- 模型评估结果 (逻辑回归) ---

[ 1. 总体准确率 ]
模型在测试集上的准确率为: 86.25%

[ 2. 详细分类报告 ]
                  precision    recall  f1-score   support

A-type (Class 0)       0.90      0.94      0.92       155
I-type (Class 1)       0.82      0.79      0.81        81
S-type (Class 2)       0.79      0.67      0.72        33

        accuracy                           0.86       269
       macro avg       0.83      0.80      0.81       269
    weighted avg       0.86      0.86      0.86       269


步骤 6/10: 正在生成混淆矩阵图表...
图表已保存为: cm_logistic_regression.png

步骤 7/10: 正在生成分类报告柱状图...
图表已保存为: report_bar_chart_lr.png

步骤 8/10: 正在生成特征重要性柱状图...
图表已保存为: feature_importance_lr.png

步骤 9/10: 正在生成最重要特征散点图...
图表已保存为: scatter_top_features_

Font 'default' does not have a glyph for '-' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '-' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '-' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '-' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '-' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '-' [U+2212], substituting with a dummy symbol.


图表已保存为: validation_curve_lr.png

--- 流程结束 (已生成最多5个图表) ---


In [10]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import validation_curve # <-- 新增
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 忽略将来可能的警告
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 新增: 设置 Matplotlib 支持中文 ---
try:
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 尝试指定中文字体
    print("中文字体 'SimHei' 加载成功。")
except Exception as e:
    print(f"中文字体 'SimHei' 设置失败, 图表标题可能显示为方块: {e}")
    print("您可以尝试安装 'SimHei' 字体, 或在代码中更改为 'Microsoft YaHei' 等您已有的中文字体")

# 无论中文字体是否加载成功, 都 *必须* 设置这一项来修复负号显示问题
plt.rcParams['axes.unicode_minus'] = False
# --- 新增结束 ---


print("--- 机器学习模型优化 (随机森林) ---")

try:
    # 1. 加载您上一步生成的处理后数据
    print("步骤 1/10: 正在加载 'train_processed.csv' 和 'test_processed.csv'...")
    train_df = pd.read_csv("train_processed.csv")
    test_df = pd.read_csv("test_processed.csv")
    print("数据加载完毕。")

    # 2. 再次分离特征 (X) 和目标 (y)
    print("步骤 2/10: 正在分离特征 (X) 和目标 (y)...")
    target_column = 'Type_Encoded'
    
    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]
    
    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]
    
    # 合并训练集和测试集 (用于绘图)
    X_all = pd.concat([X_train, X_test], axis=0)
    y_all = pd.concat([y_train, y_test], axis=0)
    
    # 获取特征名称列表，以便后续显示“重要特征”
    feature_names = X_train.columns
    class_names_plot = ['A-type', 'I-type', 'S-type'] # 0, 1, 2
    
    print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")

    # 3. 初始化和训练模型（随机森林）
    print("步骤 3/10: 正在初始化随机森林分类器...")
    rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    
    print("步骤 4/10: 正在使用训练数据 (X_train) 训练模型...")
    rf_model.fit(X_train, y_train)
    print("模型训练完成！")

    # 4. 评估模型
    print("步骤 5/10: 正在使用测试数据 (X_test) 评估模型性能...")
    y_pred_rf = rf_model.predict(X_test)
    accuracy_rf = accuracy_score(y_test, y_pred_rf)
    
    print(f"\n--- 模型评估结果 (随机森林) ---")
    print(f"\n[ 1. 总体准确率 ]")
    print(f"模型在测试集上的准确率为: {accuracy_rf * 100:.2f}%")
    
    target_names_report = ['A-type (Class 0)', 'I-type (Class 1)', 'S-type (Class 2)']
    report_rf = classification_report(y_test, y_pred_rf, target_names=target_names_report)
    
    print(f"\n[ 2. 详细分类报告 ]\n{report_rf}")

    # --- 6. 绘制混淆矩阵 ---
    print("\n步骤 6/10: 正在生成混淆矩阵图表...")
    cm = confusion_matrix(y_test, y_pred_rf)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', # 换个颜色
                xticklabels=class_names_plot, yticklabels=class_names_plot)
    plt.title(f'随机森林 混淆矩阵\n(总体准确率: {accuracy_rf * 100:.2f}%)')
    plt.ylabel('真实类别 (True Label)')
    plt.xlabel('预测类别 (Predicted Label)')
    plot_filename_cm = "cm_random_forest.png"
    plt.tight_layout()
    plt.savefig(plot_filename_cm)
    print(f"图表已保存为: {plot_filename_cm}")
    plt.close()

    # --- 7. 绘制分类报告柱状图 ---
    print("\n步骤 7/10: 正在生成分类报告柱状图...")
    report_dict = classification_report(y_test, y_pred_rf, target_names=class_names_plot, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    report_df = report_df.loc[class_names_plot, ['precision', 'recall', 'f1-score']]
    
    report_df.plot(kind='bar', figsize=(12, 7), rot=0)
    plt.title('随机森林 分类报告指标')
    plt.ylabel('得分 (Score)')
    plt.xlabel('岩石类别')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.ylim(0, 1.05)
    for p in plt.gca().patches:
        plt.gca().annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='center', xytext=(0, 9), textcoords='offset points')
    plot_filename_report = "report_bar_chart_rf.png"
    plt.tight_layout()
    plt.savefig(plot_filename_report)
    print(f"图表已保存为: {plot_filename_report}")
    plt.close()

    # --- 8. 绘制特征重要性柱状图 ---
    print("\n步骤 8/10: 正在生成特征重要性柱状图...")
    importances = rf_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(12, 9))
    sns.barplot(x='Importance', y='Feature', data=feature_importance_df.head(15), palette='viridis')
    plt.title('随机森林 - Top 15 重要特征')
    plt.xlabel('重要性得分 (Importance Score)')
    plt.ylabel('特征 (Feature)')
    
    plot_filename_features = "feature_importance_rf.png"
    plt.tight_layout()
    plt.savefig(plot_filename_features)
    print(f"图表已保存为: {plot_filename_features}")
    plt.close()

    # --- 9. 新增: 绘制特征散点图 ---
    print("\n步骤 9/10: 正在生成最重要特征散点图...")
    # 从上一步获取最重要的两个特征
    top_feature_1 = feature_importance_df.iloc[0]['Feature']
    top_feature_2 = feature_importance_df.iloc[1]['Feature']
    
    # 将 y (0,1,2) 映射回类别名称 (A, I, S)
    y_all_named = y_all.map({0: 'A-type', 1: 'I-type', 2: 'S-type'})
    
    # 创建包含绘图所需数据的 DataFrame
    scatter_df = pd.DataFrame({
        'Feature_1': X_all[top_feature_1],
        'Feature_2': X_all[top_feature_2],
        'Type': y_all_named
    })

    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=scatter_df, x='Feature_1', y='Feature_2', hue='Type', alpha=0.7, palette='bright')
    plt.title(f'随机森林 最重要特征散点图\n({top_feature_1} vs {top_feature_2})')
    plt.xlabel(top_feature_1)
    plt.ylabel(top_feature_2)
    
    plot_filename_scatter = "scatter_top_features_rf.png"
    plt.tight_layout()
    plt.savefig(plot_filename_scatter)
    print(f"图表已保存为: {plot_filename_scatter}")
    plt.close()

    # --- 10. 新增: 绘制验证曲线 (折线图) ---
    print("\n步骤 10/10: 正在生成验证曲线(折线图)... (这可能需要一点时间)")
    try:
        # 'max_depth' 是随机森林的一个关键参数
        param_range = [1, 3, 5, 7, 10, 15, 20] # 测试 7 个不同的树深度
        
        train_scores, test_scores = validation_curve(
            RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1), # 用 50 棵树加速计算
            X_all, y_all, # 使用所有数据来绘制曲线
            param_name="max_depth", # 我们要调整的参数
            param_range=param_range,
            cv=3, # 3折交叉验证
            scoring="accuracy", # 评估指标
            n_jobs=1 # 【*** 关键修复 ***】 改为 1 (串行计算), 避免内存崩溃
        )
        
        # 计算平均值和标准差
        train_scores_mean = np.mean(train_scores, axis=1)
        train_scores_std = np.std(train_scores, axis=1)
        test_scores_mean = np.mean(test_scores, axis=1)
        test_scores_std = np.std(test_scores, axis=1)

        plt.figure(figsize=(10, 6))
        plt.title("随机森林 验证曲线 (参数 'max_depth')")
        plt.xlabel("树的最大深度 (max_depth) (值越大, 模型越复杂)")
        plt.ylabel("准确率 (Accuracy)")
        plt.ylim(0.0, 1.1)
        lw = 2
        
        plt.plot(param_range, train_scores_mean, label="训练得分", color="darkorange", lw=lw)
        plt.fill_between(param_range, train_scores_mean - train_scores_std,
                         train_scores_mean + train_scores_std, alpha=0.2, color="darkorange")
        
        plt.plot(param_range, test_scores_mean, label="交叉验证得分 (测试)", color="navy", lw=lw)
        plt.fill_between(param_range, test_scores_mean - test_scores_std,
                         test_scores_mean + test_scores_std, alpha=0.2, color="navy")
        
        plt.legend(loc="best")
        
        plot_filename_validation = "validation_curve_rf.png"
        plt.tight_layout()
        plt.savefig(plot_filename_validation)
        print(f"图表已保存为: {plot_filename_validation}")
        plt.close()

    except Exception as e_val:
        print(f"验证曲线绘制失败: {e_val}")
    # --- 新Z结束 ---

    print("\n--- 流程结束 (已生成最多5个图表) ---")

except FileNotFoundError:
    print("\n--- 错误: 文件未找到 ---")
    print("找不到 'train_processed.csv' 或 'test_processed.csv'。")
except ImportError:
    print(f"\n--- 错误: 缺少库 ---")
    print("请确保已安装 'matplotlib' 和 'seaborn' 库。")
except Exception as e:
    print(f"\n--- 处理过程中发生错误 ---")
    print(f"错误详情: {e}")


中文字体 'SimHei' 加载成功。
--- 机器学习模型优化 (随机森林) ---
步骤 1/10: 正在加载 'train_processed.csv' 和 'test_processed.csv'...
数据加载完毕。
步骤 2/10: 正在分离特征 (X) 和目标 (y)...
训练集: (1072, 46), 测试集: (269, 46)
步骤 3/10: 正在初始化随机森林分类器...
步骤 4/10: 正在使用训练数据 (X_train) 训练模型...
模型训练完成！
步骤 5/10: 正在使用测试数据 (X_test) 评估模型性能...

--- 模型评估结果 (随机森林) ---

[ 1. 总体准确率 ]
模型在测试集上的准确率为: 89.59%

[ 2. 详细分类报告 ]
                  precision    recall  f1-score   support

A-type (Class 0)       0.92      0.95      0.93       155
I-type (Class 1)       0.85      0.83      0.84        81
S-type (Class 2)       0.90      0.82      0.86        33

        accuracy                           0.90       269
       macro avg       0.89      0.86      0.88       269
    weighted avg       0.90      0.90      0.90       269


步骤 6/10: 正在生成混淆矩阵图表...
图表已保存为: cm_random_forest.png

步骤 7/10: 正在生成分类报告柱状图...
图表已保存为: report_bar_chart_rf.png

步骤 8/10: 正在生成特征重要性柱状图...
图表已保存为: feature_importance_rf.png

步骤 9/10: 正在生成最重要特征散点图...
图表已保存为: scatter_top_features_rf.png

步骤

In [11]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE  # <-- 导入 SMOTE
from sklearn.model_selection import validation_curve # <-- 新增
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 忽略将来可能的警告
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 新增: 设置 Matplotlib 支持中文 ---
try:
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 尝试指定中文字体
    print("中文字体 'SimHei' 加载成功。")
except Exception as e:
    print(f"中文字体 'SimHei' 设置失败, 图表标题可能显示为方块: {e}")
    print("您可以尝试安装 'SimHei' 字体, 或在代码中更改为 'Microsoft YaHei' 等您已有的中文字体")

# 无论中文字体是否加载成功, 都 *必须* 设置这一项来修复负号显示问题
plt.rcParams['axes.unicode_minus'] = False
# --- 新增结束 ---


print("--- 机器学习高级步骤 (SMOTE + 随机森林) ---")

try:
    # 1. 加载您上一步生成的处理后数据
    print("步骤 1/11: 正在加载 'train_processed.csv' 和 'test_processed.csv'...")
    train_df = pd.read_csv("train_processed.csv")
    test_df = pd.read_csv("test_processed.csv")
    print("数据加载完毕。")

    # 2. 再次分离特征 (X) 和目标 (y)
    print("步骤 2/11: 正在分离特征 (X) 和目标 (y)...")
    target_column = 'Type_Encoded'
    
    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]
    
    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]
    
    # 合并训练集和测试集 (用于绘图)
    X_all = pd.concat([X_train, X_test], axis=0)
    y_all = pd.concat([y_train, y_test], axis=0)
    
    # 获取特征名称列表，以便后续显示“重要特征”
    feature_names = X_train.columns
    class_names_plot = ['A-type', 'I-type', 'S-type'] # 0, 1, 2
    
    print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")
    
    # 打印 SMOTE 前的样本分布
    print("\n--- SMOTE 处理前 ---")
    print(f"原始训练集样本分布:\n{y_train.value_counts().sort_index()}")
    print("(0=A-type, 1=I-type, 2=S-type)")


    # 3. 【新步骤】应用 SMOTE 来平衡训练数据
    print("\n步骤 3/11: 正在应用 SMOTE 来平衡训练数据...")
    smote = SMOTE(random_state=42, n_jobs=-1)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    
    print("\n--- SMOTE 处理后 ---")
    print(f"新训练集样本分布:\n{y_train_smote.value_counts().sort_index()}")
    print("所有类别的样本数已变得相同。")


    # 4. 在 SMOTE 处理后的数据上训练模型
    print("\n步骤 4/11: 正在使用 (SMOTE) 平衡后的数据训练新模型...")
    rf_model_smote = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    
    # 使用新数据 (X_train_smote, y_train_smote) 进行训练
    rf_model_smote.fit(X_train_smote, y_train_smote)
    print("模型训练完成。")

    # 5. 在 *原始* 测试集上评估模型
    print("\n步骤 5/11: 正在使用 *原始* 测试集评估 (SMOTE) 模型...")
    # 评估必须在原始的、真实的 X_test 上进行
    y_pred_smote = rf_model_smote.predict(X_test)
    
    accuracy_smote = accuracy_score(y_test, y_pred_smote)
    print(f"\n--- 模型评估结果 (SMOTE + 随机森林) ---")
    print(f"\n[ 1. 总体准确率 ]")
    print(f"模型 (SMOTE) 准确率为: {accuracy_smote * 100:.2f}%")
    
    # 6. 打印详细的分类报告
    print("\n[ 2. 详细分类报告 ]")
    target_names_report = ['A-type (Class 0)', 'I-type (Class 1)', 'S-type (Class 2)']
    report_smote = classification_report(y_test, y_pred_smote, target_names=target_names_report)
    print(report_smote)

    # --- 7. 绘制混淆矩阵 ---
    print("\n步骤 7/11: 正在生成混淆矩阵图表...")
    cm = confusion_matrix(y_test, y_pred_smote)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='PuBu', # 换个颜色 (Purple-Blue)
                xticklabels=class_names_plot, yticklabels=class_names_plot)
    plt.title(f'SMOTE + 随机森林 混淆矩阵\n(总体准确率: {accuracy_smote * 100:.2f}%)')
    plt.ylabel('真实类别 (True Label)')
    plt.xlabel('预测类别 (Predicted Label)')
    plot_filename_cm = "cm_smote_forest.png"
    plt.tight_layout()
    plt.savefig(plot_filename_cm)
    print(f"图表已保存为: {plot_filename_cm}")
    plt.close()

    # --- 8. 绘制分类报告柱状图 ---
    print("\n步骤 8/11: 正在生成分类报告柱状图...")
    report_dict = classification_report(y_test, y_pred_smote, target_names=class_names_plot, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    report_df = report_df.loc[class_names_plot, ['precision', 'recall', 'f1-score']]
    
    report_df.plot(kind='bar', figsize=(12, 7), rot=0)
    plt.title('SMOTE + 随机森林 分类报告指标')
    plt.ylabel('得分 (Score)')
    plt.xlabel('岩石类别')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.ylim(0, 1.05)
    for p in plt.gca().patches:
        plt.gca().annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='center', xytext=(0, 9), textcoords='offset points')
    plot_filename_report = "report_bar_chart_smote.png"
    plt.tight_layout()
    plt.savefig(plot_filename_report)
    print(f"图表已保存为: {plot_filename_report}")
    plt.close()

    # --- 9. 绘制特征重要性柱状图 ---
    print("\n步骤 9/11: 正在生成特征重要性柱状图 (SMOTE 模型)...")
    # 注意: 我们使用的是 SMOTE 训练后的 rf_model_smote
    importances_smote = rf_model_smote.feature_importances_
    feature_importance_smote_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances_smote
    }).sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(12, 9))
    sns.barplot(x='Importance', y='Feature', data=feature_importance_smote_df.head(15), palette='cividis') # 换个颜色
    plt.title('SMOTE + 随机森林 - Top 15 重要特征')
    plt.xlabel('重要性得分 (Importance Score)')
    plt.ylabel('特征 (Feature)')
    
    plot_filename_features = "feature_importance_smote.png"
    plt.tight_layout()
    plt.savefig(plot_filename_features)
    print(f"图表已保存为: {plot_filename_features}")
    plt.close()

    # --- 10. 新增: 绘制特征散点图 ---
    print("\n步骤 10/11: 正在生成最重要特征散点图 (SMOTE 模型)...")
    # 从上一步获取最重要的两个特征
    top_feature_1 = feature_importance_smote_df.iloc[0]['Feature']
    top_feature_2 = feature_importance_smote_df.iloc[1]['Feature']
    
    # 将 y (0,1,2) 映射回类别名称 (A, I, S)
    y_all_named = y_all.map({0: 'A-type', 1: 'I-type', 2: 'S-type'})
    
    # 创建包含绘图所需数据的 DataFrame (使用 *原始* X_all 数据)
    scatter_df = pd.DataFrame({
        'Feature_1': X_all[top_feature_1],
        'Feature_2': X_all[top_feature_2],
        'Type': y_all_named
    })

    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=scatter_df, x='Feature_1', y='Feature_2', hue='Type', alpha=0.7, palette='bright')
    plt.title(f'SMOTE 模型 最重要特征散点图\n({top_feature_1} vs {top_feature_2})')
    plt.xlabel(top_feature_1)
    plt.ylabel(top_feature_2)
    
    plot_filename_scatter = "scatter_top_features_smote.png"
    plt.tight_layout()
    plt.savefig(plot_filename_scatter)
    print(f"图表已保存为: {plot_filename_scatter}")
    plt.close()

    # --- 11. 新增: 绘制验证曲线 (折线图) ---
    print("\n步骤 11/11: 正在生成验证曲线(折线图)... (这可能需要一点时间)")
    # 注意: 验证曲线通常在 *原始* 数据集上运行, 以分析算法对真实数据分布的超参数敏感度
    try:
        param_range = [1, 3, 5, 7, 10, 15, 20] # 测试 7 个不同的树深度
        
        train_scores, test_scores = validation_curve(
            RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1), # 用 50 棵树加速计算
            X_all, y_all, # 使用 *所有原始数据* 来绘制曲线
            param_name="max_depth", # 我们要调整的参数
            param_range=param_range,
            cv=3, # 3折交叉验证
            scoring="accuracy", # 评估指标
            n_jobs=1 # 【*** 关键修复 ***】 改为 1 (串行计算), 避免内存崩溃
        )
        
        # 计算平均值和标准差
        train_scores_mean = np.mean(train_scores, axis=1)
        train_scores_std = np.std(train_scores, axis=1)
        test_scores_mean = np.mean(test_scores, axis=1)
        test_scores_std = np.std(test_scores, axis=1)

        plt.figure(figsize=(10, 6))
        plt.title("随机森林 验证曲线 (参数 'max_depth')")
        plt.xlabel("树的最大深度 (max_depth) (值越大, 模型越复杂)")
        plt.ylabel("准确率 (Accuracy)")
        plt.ylim(0.0, 1.1)
        lw = 2
        
        plt.plot(param_range, train_scores_mean, label="训练得分", color="darkorange", lw=lw)
        plt.fill_between(param_range, train_scores_mean - train_scores_std,
                         train_scores_mean + train_scores_std, alpha=0.2, color="darkorange")
        
        plt.plot(param_range, test_scores_mean, label="交叉验证得分 (测试)", color="navy", lw=lw)
        plt.fill_between(param_range, test_scores_mean - test_scores_std,
                         test_scores_mean + test_scores_std, alpha=0.2, color="navy")
        
        plt.legend(loc="best")
        
        # 我们复用上一份代码的验证曲线图，因为它是一个通用分析
        plot_filename_validation = "validation_curve_rf.png" 
        plt.tight_layout()
        plt.savefig(plot_filename_validation)
        print(f"图表已保存为: {plot_filename_validation} (此图分析模型通用特性)")
        plt.close()

    except Exception as e_val:
        print(f"验证曲线绘制失败: {e_val}")
    # --- 新增结束 ---


    print("\n--- 流程结束 (已生成最多5个图表) ---")

except FileNotFoundError:
    print("\n--- 错误: 文件未找到 ---")
    print("找不到 'train_processed.csv' 或 'test_processed.csv'。")
except ImportError:
    print(f"\n--- 错误: 缺少库 ---")
    print("请先安装 'imbalanced-learn', 'matplotlib' 和 'seaborn' 库。")
except Exception as e:
    print(f"\n--- 处理过程中发生错误 ---")
    print(f"错误详情: {e}")


中文字体 'SimHei' 加载成功。
--- 机器学习高级步骤 (SMOTE + 随机森林) ---
步骤 1/11: 正在加载 'train_processed.csv' 和 'test_processed.csv'...
数据加载完毕。
步骤 2/11: 正在分离特征 (X) 和目标 (y)...
训练集: (1072, 46), 测试集: (269, 46)

--- SMOTE 处理前 ---
原始训练集样本分布:
0    619
1    324
2    129
Name: Type_Encoded, dtype: int64
(0=A-type, 1=I-type, 2=S-type)

步骤 3/11: 正在应用 SMOTE 来平衡训练数据...

--- SMOTE 处理后 ---
新训练集样本分布:
0    619
1    619
2    619
Name: Type_Encoded, dtype: int64
所有类别的样本数已变得相同。

步骤 4/11: 正在使用 (SMOTE) 平衡后的数据训练新模型...
模型训练完成。

步骤 5/11: 正在使用 *原始* 测试集评估 (SMOTE) 模型...

--- 模型评估结果 (SMOTE + 随机森林) ---

[ 1. 总体准确率 ]
模型 (SMOTE) 准确率为: 90.33%

[ 2. 详细分类报告 ]
                  precision    recall  f1-score   support

A-type (Class 0)       0.93      0.94      0.93       155
I-type (Class 1)       0.88      0.83      0.85        81
S-type (Class 2)       0.84      0.94      0.89        33

        accuracy                           0.90       269
       macro avg       0.88      0.90      0.89       269
    weighted avg       0.90      0.90 

In [12]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE  # <-- 导入 SMOTE
from sklearn.model_selection import validation_curve # <-- 新增
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
# --- 新增交叉验证的导入 ---
from sklearn.model_selection import StratifiedKFold, cross_val_score
from imblearn.pipeline import Pipeline as ImbPipeline # <-- 使用 imblearn 的 Pipeline
# --- 导入结束 ---

# 忽略将来可能的警告
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 新增: 设置 Matplotlib 支持中文 ---
try:
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 尝试指定中文字体
    print("中文字体 'SimHei' 加载成功。")
except Exception as e:
    print(f"中文字体 'SimHei' 设置失败, 图表标题可能显示为方块: {e}")
    print("您可以尝试安装 'SimHei' 字体, 或在代码中更改为 'Microsoft YaHei' 等您已有的中文字体")

# 无论中文字体是否加载成功, 都 *必须* 设置这一项来修复负号显示问题
plt.rcParams['axes.unicode_minus'] = False
# --- 新增结束 ---


print("--- 机器学习高级步骤 (SMOTE + 随机森林 + 交叉验证) ---")

try:
    # 1. 加载您上一步生成的处理后数据
    print("步骤 1/12: 正在加载 'train_processed.csv' 和 'test_processed.csv'...")
    train_df = pd.read_csv("train_processed.csv")
    test_df = pd.read_csv("test_processed.csv")
    print("数据加载完毕。")

    # 2. 再次分离特征 (X) 和目标 (y)
    print("步骤 2/12: 正在分离特征 (X) 和目标 (y)...")
    target_column = 'Type_Encoded'
    
    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]
    
    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]
    
    # 合并训练集和测试集 (用于 交叉验证 和 绘图)
    # 我们将在 *所有* 数据上运行交叉验证，以获得最可靠的评估
    X_all = pd.concat([X_train, X_test], axis=0)
    y_all = pd.concat([y_train, y_test], axis=0)
    
    # 获取特征名称列表，以便后续显示“重要特征”
    feature_names = X_train.columns
    class_names_plot = ['A-type', 'I-type', 'S-type'] # 0, 1, 2
    
    print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}, 全部数据: {X_all.shape}")
    
    # 打印 SMOTE 前的样本分布
    print("\n--- SMOTE 处理前 (训练集) ---")
    print(f"原始训练集样本分布:\n{y_train.value_counts().sort_index()}")
    print("(0=A-type, 1=I-type, 2=S-type)")


    # 3. 【新步骤】应用 SMOTE 来平衡训练数据
    print("\n步骤 3/12: 正在应用 SMOTE 来平衡 *单次* 训练数据...")
    smote = SMOTE(random_state=42, n_jobs=-1)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    
    print("\n--- SMOTE 处理后 ---")
    print(f"新训练集样本分布:\n{y_train_smote.value_counts().sort_index()}")
    print("所有类别的样本数已变得相同。")


    # 4. 在 SMOTE 处理后的数据上训练模型 (用于单次评估)
    print("\n步骤 4/12: 正在使用 (SMOTE) 平衡后的数据训练 *单次* 新模型...")
    rf_model_smote = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    
    # 使用新数据 (X_train_smote, y_train_smote) 进行训练
    rf_model_smote.fit(X_train_smote, y_train_smote)
    print("模型训练完成。")

    # 5. 在 *原始* 测试集上评估模型 (单次评估)
    print("\n步骤 5/12: 正在使用 *原始* 测试集评估 (SMOTE) 模型...")
    y_pred_smote = rf_model_smote.predict(X_test)
    
    accuracy_smote = accuracy_score(y_test, y_pred_smote)
    print(f"\n--- 模型评估结果 (SMOTE + 随机森林) [单次 80/20 拆分] ---")
    print(f"\n[ 1. 总体准确率 ]")
    print(f"模型 (SMOTE) 准确率为: {accuracy_smote * 100:.2f}%")
    
    # 6. 打印详细的分类报告 (单次评估)
    print("\n[ 2. 详细分类报告 ]")
    target_names_report = ['A-type (Class 0)', 'I-type (Class 1)', 'S-type (Class 2)']
    report_smote = classification_report(y_test, y_pred_smote, target_names=target_names_report)
    print(report_smote)

    # --- 7. 绘制混淆矩阵 (单次评估) ---
    print("\n步骤 7/12: 正在生成混淆矩阵图表...")
    cm = confusion_matrix(y_test, y_pred_smote)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='PuBu',
                xticklabels=class_names_plot, yticklabels=class_names_plot)
    plt.title(f'SMOTE + 随机森林 混淆矩阵\n(单次拆分准确率: {accuracy_smote * 100:.2f}%)')
    plt.ylabel('真实类别 (True Label)')
    plt.xlabel('预测类别 (Predicted Label)')
    plot_filename_cm = "cm_smote_forest.png"
    plt.tight_layout()
    plt.savefig(plot_filename_cm)
    print(f"图表已保存为: {plot_filename_cm}")
    plt.close()

    # --- 8. 绘制分类报告柱状图 (单次评估) ---
    print("\n步骤 8/12: 正在生成分类报告柱状图...")
    report_dict = classification_report(y_test, y_pred_smote, target_names=class_names_plot, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    report_df = report_df.loc[class_names_plot, ['precision', 'recall', 'f1-score']]
    
    report_df.plot(kind='bar', figsize=(12, 7), rot=0)
    plt.title('SMOTE + 随机森林 分类报告指标 (单次拆分)')
    plt.ylabel('得分 (Score)')
    plt.xlabel('岩石类别')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.ylim(0, 1.05)
    for p in plt.gca().patches:
        plt.gca().annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='center', xytext=(0, 9), textcoords='offset points')
    plot_filename_report = "report_bar_chart_smote.png"
    plt.tight_layout()
    plt.savefig(plot_filename_report)
    print(f"图表已保存为: {plot_filename_report}")
    plt.close()

    # --- 9. 绘制特征重要性柱状图 (单次评估) ---
    print("\n步骤 9/12: 正在生成特征重要性柱状图 (SMOTE 模型)...")
    importances_smote = rf_model_smote.feature_importances_
    feature_importance_smote_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances_smote
    }).sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(12, 9))
    sns.barplot(x='Importance', y='Feature', data=feature_importance_smote_df.head(15), palette='cividis')
    plt.title('SMOTE + 随机森林 - Top 15 重要特征 (单次训练)')
    plt.xlabel('重要性得分 (Importance Score)')
    plt.ylabel('特征 (Feature)')
    
    plot_filename_features = "feature_importance_smote.png"
    plt.tight_layout()
    plt.savefig(plot_filename_features)
    print(f"图表已保存为: {plot_filename_features}")
    plt.close()

    # --- 10. 绘制特征散点图 (单次评估) ---
    print("\n步骤 10/12: 正在生成最重要特征散点图 (SMOTE 模型)...")
    top_feature_1 = feature_importance_smote_df.iloc[0]['Feature']
    top_feature_2 = feature_importance_smote_df.iloc[1]['Feature']
    
    y_all_named = y_all.map({0: 'A-type', 1: 'I-type', 2: 'S-type'})
    
    scatter_df = pd.DataFrame({
        'Feature_1': X_all[top_feature_1],
        'Feature_2': X_all[top_feature_2],
        'Type': y_all_named
    })

    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=scatter_df, x='Feature_1', y='Feature_2', hue='Type', alpha=0.7, palette='bright')
    plt.title(f'SMOTE 模型 最重要特征散点图\n({top_feature_1} vs {top_feature_2})')
    plt.xlabel(top_feature_1)
    plt.ylabel(top_feature_2)
    
    plot_filename_scatter = "scatter_top_features_smote.png"
    plt.tight_layout()
    plt.savefig(plot_filename_scatter)
    print(f"图表已保存为: {plot_filename_scatter}")
    plt.close()

    # --- 11. 绘制验证曲线 (折线图) ---
    print("\n步骤 11/12: 正在生成验证曲线(折线图)... (这可能需要一点时间)")
    try:
        param_range = [1, 3, 5, 7, 10, 15, 20] 
        train_scores, test_scores = validation_curve(
            RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1),
            X_all, y_all,
            param_name="max_depth",
            param_range=param_range,
            cv=3,
            scoring="accuracy",
            n_jobs=1 # 修复
        )
        
        train_scores_mean = np.mean(train_scores, axis=1)
        train_scores_std = np.std(train_scores, axis=1)
        test_scores_mean = np.mean(test_scores, axis=1)
        test_scores_std = np.std(test_scores, axis=1)

        plt.figure(figsize=(10, 6))
        plt.title("随机森林 验证曲线 (参数 'max_depth')")
        plt.xlabel("树的最大深度 (max_depth) (值越大, 模型越复杂)")
        plt.ylabel("准确率 (Accuracy)")
        plt.ylim(0.0, 1.1)
        lw = 2
        plt.plot(param_range, train_scores_mean, label="训练得分", color="darkorange", lw=lw)
        plt.fill_between(param_range, train_scores_mean - train_scores_std,
                         train_scores_mean + train_scores_std, alpha=0.2, color="darkorange")
        plt.plot(param_range, test_scores_mean, label="交叉验证得分 (测试)", color="navy", lw=lw)
        plt.fill_between(param_range, test_scores_mean - test_scores_std,
                         test_scores_mean + test_scores_std, alpha=0.2, color="navy")
        plt.legend(loc="best")
        
        plot_filename_validation = "validation_curve_rf.png" 
        plt.tight_layout()
        plt.savefig(plot_filename_validation)
        print(f"图表已保存为: {plot_filename_validation} (此图分析模型通用特性)")
        plt.close()

    except Exception as e_val:
        print(f"验证曲线绘制失败: {e_val}")
        
    # --- 12. 【*** 全新步骤: 10-折交叉验证 ***】 ---
    print("\n步骤 12/12: 正在执行 10-折交叉验证... (这会需要几分钟时间)")
    
    # 1. 创建一个包含 SMOTE 和 随机森林 的 *流水线 (Pipeline)*
    #    这能确保 SMOTE *只* 应用于每次循环的训练集
    model_pipeline = ImbPipeline([
        ('smote', SMOTE(random_state=42, n_jobs=-1)),
        ('model', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ])
    
    # 2. 定义10-折交叉验证
    #    StratifiedKFold 确保每一折中的 A/I/S-type 比例都和原始数据一致
    k_folds = 10
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
    
    # 3. 运行交叉验证
    #    我们使用 *全部* 数据 (X_all, y_all) 来进行最可靠的评估
    #    n_jobs=1 (串行) 来避免 SMOTE 和 RF 并行时内存崩溃
    try:
        cv_scores = cross_val_score(model_pipeline, X_all, y_all, cv=skf, scoring='accuracy', n_jobs=1)
        
        print("\n--- 10-折交叉验证结果 ---")
        print(f"10 次试验的准确率分数: \n{np.round(cv_scores, 4)}")
        print("\n[ 最终可信得分 ]")
        print(f"平均准确率: {np.mean(cv_scores) * 100:.2f}%")
        print(f"标准差: +/- {np.std(cv_scores) * 100:.2f}%")
        
        # 4. 绘制交叉验证得分的箱形图
        print("\n正在生成交叉验证得分箱形图...")
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=cv_scores, orient='h', palette='pastel')
        sns.stripplot(data=cv_scores, orient='h', color='black', alpha=0.7, jitter=0.1)
        plt.axvline(np.mean(cv_scores), color='red', linestyle='--', label=f'平均值: {np.mean(cv_scores):.4f}')
        plt.title(f'10-折交叉验证得分分布 (SMOTE + 随机森林)')
        plt.xlabel('准确率 (Accuracy)')
        plt.xlim(0.8, 1.0) # 将 X 轴放大到 80% - 100%
        plt.legend()
        
        plot_filename_cv = "cv_scores_boxplot.png"
        plt.tight_layout()
        plt.savefig(plot_filename_cv)
        print(f"图表已保存为: {plot_filename_cv}")
        plt.close()

    except Exception as e_cv:
        print(f"交叉验证执行失败: {e_cv}")
    # --- 交叉验证结束 ---

    print("\n--- 流程结束 (已生成最多6个图表) ---")

except FileNotFoundError:
    print("\n--- 错误: 文件未找到 ---")
    print("找不到 'train_processed.csv' 或 'test_processed.csv'。")
except ImportError:
    print(f"\n--- 错误: 缺少库 ---")
    print("请先安装 'imbalanced-learn', 'matplotlib' 和 'seaborn' 库。")
except Exception as e:
    print(f"\n--- 处理过程中发生错误 ---")
    print(f"错误详情: {e}")



中文字体 'SimHei' 加载成功。
--- 机器学习高级步骤 (SMOTE + 随机森林 + 交叉验证) ---
步骤 1/12: 正在加载 'train_processed.csv' 和 'test_processed.csv'...
数据加载完毕。
步骤 2/12: 正在分离特征 (X) 和目标 (y)...
训练集: (1072, 46), 测试集: (269, 46), 全部数据: (1341, 46)

--- SMOTE 处理前 (训练集) ---
原始训练集样本分布:
0    619
1    324
2    129
Name: Type_Encoded, dtype: int64
(0=A-type, 1=I-type, 2=S-type)

步骤 3/12: 正在应用 SMOTE 来平衡 *单次* 训练数据...

--- SMOTE 处理后 ---
新训练集样本分布:
0    619
1    619
2    619
Name: Type_Encoded, dtype: int64
所有类别的样本数已变得相同。

步骤 4/12: 正在使用 (SMOTE) 平衡后的数据训练 *单次* 新模型...
模型训练完成。

步骤 5/12: 正在使用 *原始* 测试集评估 (SMOTE) 模型...

--- 模型评估结果 (SMOTE + 随机森林) [单次 80/20 拆分] ---

[ 1. 总体准确率 ]
模型 (SMOTE) 准确率为: 90.33%

[ 2. 详细分类报告 ]
                  precision    recall  f1-score   support

A-type (Class 0)       0.93      0.94      0.93       155
I-type (Class 1)       0.88      0.83      0.85        81
S-type (Class 2)       0.84      0.94      0.89        33

        accuracy                           0.90       269
       macro avg       0.88      0.90  

In [13]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE  # <-- 修正了这里的导入
from sklearn.model_selection import validation_curve # <-- 新增
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
# --- 新增交叉验证和网格搜索的导入 ---
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline # <-- 使用 imblearn 的 Pipeline
# --- 导入结束 ---


# 忽略将来可能的警告
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 新增: 设置 Matplotlib 支持中文 ---
try:
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 尝试指定中文字体
    print("中文字体 'SimHei' 加载成功。")
except Exception as e:
    print(f"中文字体 'SimHei' 设置失败, 图表标题可能显示为方块: {e}")

# 无论中文字体是否加载成功, 都 *必须* 设置这一项来修复负号显示问题
plt.rcParams['axes.unicode_minus'] = False
# --- 新增结束 ---


print("--- 机器学习高级步骤 (SMOTE + 随机森林 + 交叉验证 + 调优) ---")

try:
    # 1. 加载您上一步生成的处理后数据
    print("步骤 1/13: 正在加载 'train_processed.csv' 和 'test_processed.csv'...")
    train_df = pd.read_csv("train_processed.csv")
    test_df = pd.read_csv("test_processed.csv")
    print("数据加载完毕。")

    # 2. 再次分离特征 (X) 和目标 (y)
    print("步骤 2/13: 正在分离特征 (X) 和目标 (y)...")
    target_column = 'Type_Encoded'
    
    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]
    
    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]
    
    # --- 合并训练集和测试集 (用于 交叉验证 和 绘图) ---
    X_all = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
    y_all = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)
    # --- 结束 ---
    
    # 获取特征名称列表，以便后续显示“重要特征”
    feature_names = X_train.columns
    class_names_plot = ['A-type', 'I-type', 'S-type'] # 0, 1, 2
    
    print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}, 全部数据: {X_all.shape}")
    
    # 打印 SMOTE 前的样本分布
    print("\n--- SMOTE 处理前 (训练集) ---")
    print(f"原始训练集样本分布:\n{y_train.value_counts().sort_index()}")
    print("(0=A-type, 1=I-type, 2=S-type)")


    # 3. 【新步骤】应用 SMOTE 来平衡训练数据
    print("\n步骤 3/13: 正在应用 SMOTE 来平衡 *单次* 训练数据...")
    # n_jobs=-1 会使用所有CPU核心加速
    smote = SMOTE(random_state=42, n_jobs=-1)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    
    print("\n--- SMOTE 处理后 ---")
    print(f"新训练集样本分布:\n{y_train_smote.value_counts().sort_index()}")
    print("所有类别的样本数已变得相同。")


    # 4. 在 SMOTE 处理后的数据上训练模型
    print("\n步骤 4/13: 正在使用 (SMOTE) 平衡后的数据训练 *单次* 新模型...")
    rf_model_smote = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    
    # 使用新数据 (X_train_smote, y_train_smote) 进行训练
    rf_model_smote.fit(X_train_smote, y_train_smote)
    print("模型训练完成。")

    # 5. 在 *原始* 测试集上评估模型
    print("\n步骤 5/13: 正在使用 *原始* 测试集评估 (SMOTE) 模型...")
    # 评估必须在原始的、真实的 X_test 上进行
    y_pred_smote = rf_model_smote.predict(X_test)
    
    accuracy_smote = accuracy_score(y_test, y_pred_smote)
    print(f"\n--- 模型评估结果 (SMOTE + 随机森林) [单次 80/20 拆分] ---")
    print(f"\n[ 1. 总体准确率 ]")
    print(f"模型 (SMOTE) 准确率为: {accuracy_smote * 100:.2f}%")
    
    # 6. 打印详细的分类报告
    print("\n[ 2. 详细分类报告 ]")
    target_names_report = ['A-type (Class 0)', 'I-type (Class 1)', 'S-type (Class 2)']
    report_smote = classification_report(y_test, y_pred_smote, target_names=target_names_report)
    print(report_smote)

    # --- 7. 新增: 绘制混淆矩阵 ---
    print("\n步骤 7/13: 正在生成混淆矩阵图表...")
    cm = confusion_matrix(y_test, y_pred_smote)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='PuBu', # 换个颜色 (Purple-Blue)
                xticklabels=class_names_plot, yticklabels=class_names_plot)
    plt.title(f'SMOTE + 随机森林 混淆矩阵\n(总体准确率: {accuracy_smote * 100:.2f}%)')
    plt.ylabel('真实类别 (True Label)')
    plt.xlabel('预测类别 (Predicted Label)')
    plot_filename_cm = "cm_smote_forest.png"
    plt.tight_layout()
    plt.savefig(plot_filename_cm)
    print(f"图表已保存为: {plot_filename_cm}")
    plt.close() # 关闭图表，防止在 jupyter 中重叠显示

    # --- 8. 新增: 绘制分类报告柱状图 ---
    print("\n步骤 8/13: 正在生成分类报告柱状图...")
    # 将分类报告转换为字典
    report_dict = classification_report(y_test, y_pred_smote, target_names=class_names_plot, output_dict=True)
    # 转换为 DataFrame 并只保留三个类别的 P, R, F1
    report_df = pd.DataFrame(report_dict).transpose()
    report_df = report_df.loc[class_names_plot, ['precision', 'recall', 'f1-score']]
    
    report_df.plot(kind='bar', figsize=(12, 7), rot=0)
    plt.title('SMOTE + 随机森林 分类报告指标')
    plt.ylabel('得分 (Score)')
    plt.xlabel('岩石类别')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.ylim(0, 1.05)
    # 在柱子上添加数值标签
    for p in plt.gca().patches:
        plt.gca().annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='center', xytext=(0, 9), textcoords='offset points')
    plot_filename_report = "report_bar_chart_smote.png"
    plt.tight_layout()
    plt.savefig(plot_filename_report)
    print(f"图表已保存为: {plot_filename_report}")
    plt.close()

    # --- 9. 新增: 绘制特征重要性柱状图 ---
    print("\n步骤 9/13: 正在生成特征重要性柱状图 (SMOTE 模型)...")
    # 注意: 我们使用的是 SMOTE 训练后的 rf_model_smote
    importances_smote = rf_model_smote.feature_importances_
    feature_importance_smote_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances_smote
    }).sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(12, 9))
    sns.barplot(x='Importance', y='Feature', data=feature_importance_smote_df.head(15), palette='cividis') # 换个颜色
    plt.title('SMOTE + 随机森林 - Top 15 重要特征')
    plt.xlabel('重要性得分 (Importance Score)')
    plt.ylabel('特征 (Feature)')
    
    plot_filename_features = "feature_importance_smote.png"
    plt.tight_layout()
    plt.savefig(plot_filename_features)
    print(f"图表已保存为: {plot_filename_features}")
    plt.close()

    # --- 10. 新增: 绘制特征散点图 ---
    print("\n步骤 10/13: 正在生成最重要特征散点图 (SMOTE 模型)...")
    # 从上一步获取最重要的两个特征
    top_feature_1 = feature_importance_smote_df.iloc[0]['Feature']
    top_feature_2 = feature_importance_smote_df.iloc[1]['Feature']
    
    # 将 y (0,1,2) 映射回类别名称 (A, I, S)
    y_all_named = y_all.map({0: 'A-type', 1: 'I-type', 2: 'S-type'})
    
    # 创建包含绘图所需数据的 DataFrame (使用 *原始* X_all 数据)
    scatter_df = pd.DataFrame({
        'Feature_1': X_all[top_feature_1],
        'Feature_2': X_all[top_feature_2],
        'Type': y_all_named
    })

    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=scatter_df, x='Feature_1', y='Feature_2', hue='Type', alpha=0.7, palette='bright')
    plt.title(f'SMOTE 模型 最重要特征散点图\n({top_feature_1} vs {top_feature_2})')
    plt.xlabel(top_feature_1)
    plt.ylabel(top_feature_2)
    
    plot_filename_scatter = "scatter_top_features_smote.png"
    plt.tight_layout()
    plt.savefig(plot_filename_scatter)
    print(f"图表已保存为: {plot_filename_scatter}")
    plt.close()

    # --- 11. 新增: 绘制验证曲线 (折线图) ---
    print("\n步骤 11/13: 正在生成验证曲线(折线图)... (这可能需要一点时间)")
    try:
        param_range = [1, 3, 5, 7, 10, 15, 20] # 测试 7 个不同的树深度
        
        train_scores, test_scores = validation_curve(
            RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1), # 用 50 棵树加速计算
            X_all, y_all, # 使用 *所有原始数据* 来绘制曲线
            param_name="max_depth", # 我们要调整的参数
            param_range=param_range,
            cv=3, # 3折交叉验证
            scoring="accuracy", # 评估指标
            n_jobs=1 # 【*** 关键修复 ***】 改为 1 (串行计算), 避免内存崩溃
        )
        
        # 计算平均值和标准差
        train_scores_mean = np.mean(train_scores, axis=1)
        train_scores_std = np.std(train_scores, axis=1)
        test_scores_mean = np.mean(test_scores, axis=1)
        test_scores_std = np.std(test_scores, axis=1)

        plt.figure(figsize=(10, 6))
        plt.title("随机森林 验证曲线 (参数 'max_depth')")
        plt.xlabel("树的最大深度 (max_depth) (值越大, 模型越复杂)")
        plt.ylabel("准确率 (Accuracy)")
        plt.ylim(0.0, 1.1)
        lw = 2
        
        plt.plot(param_range, train_scores_mean, label="训练得分", color="darkorange", lw=lw)
        plt.fill_between(param_range, train_scores_mean - train_scores_std,
                         train_scores_mean + train_scores_std, alpha=0.2, color="darkorange")
        
        plt.plot(param_range, test_scores_mean, label="交叉验证得分 (测试)", color="navy", lw=lw)
        plt.fill_between(param_range, test_scores_mean - test_scores_std,  # <-- 修复了这里的 BUG (param_Dparam_range)
                         test_scores_mean + test_scores_std, alpha=0.2, color="navy")
        
        plt.legend(loc="best")
        
        plot_filename_validation = "validation_curve_rf.png" 
        plt.tight_layout()
        plt.savefig(plot_filename_validation)
        print(f"图表已保存为: {plot_filename_validation} (此图分析模型通用特性)")
        plt.close() # 关闭图表

    except Exception as e_val:
        print(f"验证曲线绘制失败: {e_val}")
    # --- 新增结束 ---
    
    # --- 12. 【新增: 10-折交叉验证 (基准模型)】 ---
    print("\n步骤 12/13: 正在执行 10-折交叉验证 (使用 n_estimators=100)... (这会需要几分钟时间)")
    
    # 1. 创建一个包含 SMOTE 和 *默认* 随机森林 的流水线
    model_pipeline_default = ImbPipeline([
        ('smote', SMOTE(random_state=42, n_jobs=-1)),
        ('model', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
    ])
    
    # 2. 定义10-折交叉验证
    k_folds = 10
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
    
    # 3. 运行交叉验证
    default_mean_accuracy = 0.0 # 初始化
    try:
        cv_scores = cross_val_score(model_pipeline_default, X_all, y_all, cv=skf, scoring='accuracy', n_jobs=1)
        
        print("\n--- 10-折交叉验证结果 (默认参数) ---")
        print(f"10 次试验的准确率分数: \n{np.round(cv_scores, 4)}")
        print("\n[ 默认模型可信得分 ]")
        default_mean_accuracy = np.mean(cv_scores)
        print(f"平均准确率: {default_mean_accuracy * 100:.2f}%")
        print(f"标准差: +/- {np.std(cv_scores) * 100:.2f}%")
        
        # 4. 绘制交叉验证得分的箱形图
        print("\n正在生成交叉验证得分箱形图...")
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=cv_scores, orient='h', palette='pastel')
        sns.stripplot(data=cv_scores, orient='h', color='black', alpha=0.7, jitter=0.1)
        plt.axvline(np.mean(cv_scores), color='red', linestyle='--', label=f'平均值: {np.mean(cv_scores):.4f}')
        plt.title(f'10-折交叉验证得分分布 (SMOTE + 默认RF)')
        plt.xlabel('准确率 (Accuracy)')
        plt.xlim(0.8, 1.0) # 将 X 轴放大到 80% - 100%
        plt.legend()
        
        plot_filename_cv = "cv_scores_boxplot.png"
        plt.tight_layout()
        plt.savefig(plot_filename_cv)
        print(f"图表已保存为: {plot_filename_cv}")
        plt.close()

    except Exception as e_cv:
        print(f"交叉验证执行失败: {e_cv}")
    # --- 交叉验证结束 ---

    
    # --- 13. 【*** 全新步骤: 网格搜索 (Grid Search CV) ***】 ---
    print("\n步骤 13/13: 正在执行网格搜索 (Grid Search CV) 来寻找最佳参数... (这会 *非常* 耗时, 可能需要 10-20 分钟)")

    # 1. 定义我们要测试的参数网格
    #    'model__n_estimators' 中的 'model__' 是流水线中步骤的名称
    #    为节约时间, 我们只测试有限的组合
    param_grid = {
        'model__n_estimators': [100, 150],       # 尝试 2 种树的数量
        'model__max_depth': [10, 15, 20]        # 尝试 3 种树的深度
    }
    # 总共的组合数: 2 * 3 = 6 种
    # 交叉验证折数: 5 (GridSearchCV 默认是 5)
    # 总共的训练次数: 6 * 5 = 30 次
    
    # 2. 创建一个包含 SMOTE 和 随机森林 的流水线 (与交叉验证的相同)
    pipeline_for_gridsearch = ImbPipeline([
        ('smote', SMOTE(random_state=42, n_jobs=-1)),
        ('model', RandomForestClassifier(random_state=42, n_jobs=-1))
    ])

    # 3. 初始化 GridSearchCV
    #    cv=5 表示 5-折交叉验证
    #    scoring='accuracy' 是我们的评估标准
    #    n_jobs=1 (串行) 来避免内存崩溃
    grid_search = GridSearchCV(estimator=pipeline_for_gridsearch,
                               param_grid=param_grid,
                               cv=5, 
                               scoring='accuracy',
                               n_jobs=1,
                               verbose=2) # verbose=2 会打印详细的进度日志

    try:
        # 4. 在 *全部* 数据上运行网格搜索
        grid_search.fit(X_all, y_all)
        
        print("\n--- 网格搜索 (Grid Search) 结果 ---")
        print(f"\n[ 最佳得分 ]")
        best_accuracy = grid_search.best_score_
        print(f"找到的最佳平均准确率 (来自 5-折 CV): {best_accuracy * 100:.2f}%")
        if default_mean_accuracy > 0:
            print(f"(对比: 默认参数的 10-折 CV 平均准确率: {default_mean_accuracy * 100:.2f}%)")
        
        print("\n[ 最佳参数组合 ]")
        print("电脑找到的最佳参数设置是:")
        print(grid_search.best_params_)
        
        print("\n[ 性能提升 ]")
        if default_mean_accuracy > 0 and best_accuracy > default_mean_accuracy:
            print(f"通过调优, 模型性能从 {default_mean_accuracy * 100:.2f}% 提升到了 {best_accuracy * 100:.2f}%。")
        elif default_mean_accuracy > 0:
            print(f"调优后的得分 {best_accuracy * 100:.2f}% 未能超越默认参数的 {default_mean_accuracy * 100:.2f}%。")
            print("这说明我们最初 n_estimators=100 的设置已经非常接近最优解了。")
        else:
            print("已找到最佳参数。")

    except Exception as e_grid:
        print(f"网格搜索执行失败: {e_grid}")
    # --- 网格搜索结束 ---


    print("\n--- 完整流程结束 (已生成最多6个图表并完成调优) ---")

except FileNotFoundError:
    print("\n--- 错误: 文件未找到 ---")
    print("找不到 'train_processed.csv' 或 'test_processed.csv'。")
except ImportError:
    print(f"\n--- 错误: 缺少库 ---")
    print("请先安装 'imbalanced-learn', 'matplotlib' 和 'seaborn' 库。")
    print("您可以在 notebook 的一个单元格中运行: !pip install imbalanced-learn matplotlib seaborn")
except Exception as e:
    print(f"\n--- 处理过程中发生错误 ---")
    print(f"错误详情: {e}")



中文字体 'SimHei' 加载成功。
--- 机器学习高级步骤 (SMOTE + 随机森林 + 交叉验证 + 调优) ---
步骤 1/13: 正在加载 'train_processed.csv' 和 'test_processed.csv'...
数据加载完毕。
步骤 2/13: 正在分离特征 (X) 和目标 (y)...
训练集: (1072, 46), 测试集: (269, 46), 全部数据: (1341, 46)

--- SMOTE 处理前 (训练集) ---
原始训练集样本分布:
0    619
1    324
2    129
Name: Type_Encoded, dtype: int64
(0=A-type, 1=I-type, 2=S-type)

步骤 3/13: 正在应用 SMOTE 来平衡 *单次* 训练数据...

--- SMOTE 处理后 ---
新训练集样本分布:
0    619
1    619
2    619
Name: Type_Encoded, dtype: int64
所有类别的样本数已变得相同。

步骤 4/13: 正在使用 (SMOTE) 平衡后的数据训练 *单次* 新模型...
模型训练完成。

步骤 5/13: 正在使用 *原始* 测试集评估 (SMOTE) 模型...

--- 模型评估结果 (SMOTE + 随机森林) [单次 80/20 拆分] ---

[ 1. 总体准确率 ]
模型 (SMOTE) 准确率为: 90.33%

[ 2. 详细分类报告 ]
                  precision    recall  f1-score   support

A-type (Class 0)       0.93      0.94      0.93       155
I-type (Class 1)       0.88      0.83      0.85        81
S-type (Class 2)       0.84      0.94      0.89        33

        accuracy                           0.90       269
       macro avg       0.88      0

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb # <-- 导入 XGBoost
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import validation_curve
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
import time # <-- 导入 time 模块来计时

# 忽略将来可能的警告
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 设置 Matplotlib 支持中文 ---
try:
    plt.rcParams['font.sans-serif'] = ['SimHei']
    print("中文字体 'SimHei' 加载成功。")
except Exception as e:
    print(f"中文字体 'SimHei' 设置失败: {e}")
plt.rcParams['axes.unicode_minus'] = False
# --- 设置结束 ---


print("--- 机器学习“王牌”模型 (XGBoost 完整分析) ---")
start_time_total = time.time() # 记录总开始时间

try:
    # 1. 加载数据
    print("步骤 1/13: 正在加载 'train_processed.csv' 和 'test_processed.csv'...")
    train_df = pd.read_csv("train_processed.csv")
    test_df = pd.read_csv("test_processed.csv")
    print("数据加载完毕。")

    # 2. 分离特征 (X) 和目标 (y)
    print("步骤 2/13: 正在分离特征 (X) 和目标 (y)...")
    target_column = 'Type_Encoded'
    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]
    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]
    
    X_all = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)
    y_all = pd.concat([y_train, y_test], axis=0).reset_index(drop=True)
    
    feature_names = X_train.columns
    class_names_plot = ['A-type', 'I-type', 'S-type']
    print(f"全部数据: {X_all.shape}")

    # 3. SMOTE (用于单次训练)
    print("\n步骤 3/13: 正在应用 SMOTE 来平衡 *单次* 训练数据...")
    smote = SMOTE(random_state=42, n_jobs=-1)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    print(f"SMOTE 处理后训练集形状: {X_train_smote.shape}")

    # 4. 训练 XGBoost (单次)
    print("\n步骤 4/13: 正在使用 (SMOTE) 平衡后的数据训练 *单次* XGBoost 模型...")
    # 关键参数:
    # use_label_encoder=False: 禁用旧的标签编码器 (新版 XGBoost 要求)
    # eval_metric='mlogloss': 用于多分类的评估指标
    # num_class=3: 明确告知有3个类别
    xgb_model_single = xgb.XGBClassifier(
        n_estimators=100, 
        random_state=42, 
        n_jobs=-1, 
        use_label_encoder=False, 
        eval_metric='mlogloss'
    )
    xgb_model_single.fit(X_train_smote, y_train_smote)
    print("模型训练完成。")

    # 5. 评估 XGBoost (单次)
    print("\n步骤 5/13: 正在使用 *原始* 测试集评估 (XGBoost) 模型...")
    y_pred_xgb = xgb_model_single.predict(X_test)
    accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
    print(f"\n--- 模型评估结果 (SMOTE + XGBoost) [单次 80/20 拆分] ---")
    print(f"\n[ 1. 总体准确率 ]")
    print(f"模型 (XGBoost) 准确率为: {accuracy_xgb * 100:.2f}%")
    
    # 6. 分类报告 (单次)
    print("\n[ 2. 详细分类报告 ]")
    target_names_report = ['A-type (Class 0)', 'I-type (Class 1)', 'S-type (Class 2)']
    report_xgb = classification_report(y_test, y_pred_xgb, target_names=target_names_report)
    print(report_xgb)

    # 7. 混淆矩阵 (单次)
    print("\n步骤 7/13: 正在生成混淆矩阵图表...")
    cm_xgb = confusion_matrix(y_test, y_pred_xgb)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='YlGnBu', # 换个颜色 (Yellow-Green-Blue)
                xticklabels=class_names_plot, yticklabels=class_names_plot)
    plt.title(f'SMOTE + XGBoost 混淆矩阵\n(总体准确率: {accuracy_xgb * 100:.2f}%)')
    plt.ylabel('真实类别 (True Label)')
    plt.xlabel('预测类别 (Predicted Label)')
    plt.savefig("cm_xgboost.png")
    print(f"图表已保存为: cm_xgboost.png")
    plt.close()

    # 8. 分类报告柱状图 (单次)
    print("\n步骤 8/13: 正在生成分类报告柱状图...")
    report_dict_xgb = classification_report(y_test, y_pred_xgb, target_names=class_names_plot, output_dict=True)
    report_df_xgb = pd.DataFrame(report_dict_xgb).transpose().loc[class_names_plot, ['precision', 'recall', 'f1-score']]
    report_df_xgb.plot(kind='bar', figsize=(12, 7), rot=0, colormap='summer')
    plt.title('SMOTE + XGBoost 分类报告指标')
    plt.ylabel('得分 (Score)')
    plt.xlabel('岩石类别')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.ylim(0, 1.05)
    for p in plt.gca().patches:
        plt.gca().annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='center', xytext=(0, 9), textcoords='offset points')
    plt.savefig("report_bar_chart_xgb.png")
    print(f"图表已保存为: report_bar_chart_xgb.png")
    plt.close()

    # 9. 特征重要性柱状图 (单次)
    print("\n步骤 9/13: 正在生成特征重要性柱状图 (XGBoost 模型)...")
    importances_xgb = xgb_model_single.feature_importances_
    feature_importance_xgb_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances_xgb
    }).sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(12, 9))
    sns.barplot(x='Importance', y='Feature', data=feature_importance_xgb_df.head(15), palette='autumn')
    plt.title('SMOTE + XGBoost - Top 15 重要特征')
    plt.xlabel('重要性得分 (F-Score / Gain)')
    plt.ylabel('特征 (Feature)')
    plt.savefig("feature_importance_xgb.png")
    print(f"图表已保存为: feature_importance_xgb.png")
    plt.close()

    # 10. 特征散点图 (单次)
    print("\n步骤 10/13: 正在生成最重要特征散点图 (XGBoost 模型)...")
    top_feature_1_xgb = feature_importance_xgb_df.iloc[0]['Feature']
    top_feature_2_xgb = feature_importance_xgb_df.iloc[1]['Feature']
    y_all_named = y_all.map({0: 'A-type', 1: 'I-type', 2: 'S-type'})
    scatter_df_xgb = pd.DataFrame({'Feature_1': X_all[top_feature_1_xgb], 'Feature_2': X_all[top_feature_2_xgb], 'Type': y_all_named})
    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=scatter_df_xgb, x='Feature_1', y='Feature_2', hue='Type', alpha=0.7, palette='dark')
    plt.title(f'XGBoost 模型 最重要特征散点图\n({top_feature_1_xgb} vs {top_feature_2_xgb})')
    plt.xlabel(top_feature_1_xgb)
    plt.ylabel(top_feature_2_xgb)
    plt.savefig("scatter_top_features_xgb.png")
    print(f"图表已保存为: scatter_top_features_xgb.png")
    plt.close()

    # 11. 验证曲线 (折线图)
    print("\n步骤 11/13: 正在生成 XGBoost 验证曲线(折线图)... (这可能需要一点时间)")
    try:
        param_range = [1, 3, 5, 7, 10] # XGBoost 深度通常较浅
        train_scores, test_scores = validation_curve(
            xgb.XGBClassifier(n_estimators=50, random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss'),
            X_all, y_all,
            param_name="max_depth",
            param_range=param_range,
            cv=3,
            scoring="accuracy",
            n_jobs=1 
        )
        
        train_scores_mean = np.mean(train_scores, axis=1)
        test_scores_mean = np.mean(test_scores, axis=1)
        plt.figure(figsize=(10, 6))
        plt.title("XGBoost 验证曲线 (参数 'max_depth')")
        plt.xlabel("树的最大深度 (max_depth)")
        plt.ylabel("准确率 (Accuracy)")
        plt.ylim(0.0, 1.1)
        plt.plot(param_range, train_scores_mean, label="训练得分", color="r", lw=2)
        plt.plot(param_range, test_scores_mean, label="交叉验证得分 (测试)", color="g", lw=2)
        plt.legend(loc="best")
        plt.savefig("validation_curve_xgb.png")
        print(f"图表已保存为: validation_curve_xgb.png")
        plt.close()

    except Exception as e_val:
        print(f"验证曲线绘制失败: {e_val}")
    
    # --- 12. 10-折交叉验证 (XGBoost 默认参数) ---
    print("\n步骤 12/13: 正在执行 10-折交叉验证 (XGBoost 默认参数)... (这会需要几分钟时间)")
    
    model_pipeline_xgb_default = ImbPipeline([
        ('smote', SMOTE(random_state=42, n_jobs=-1)),
        ('model', xgb.XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss'))
    ])
    
    k_folds = 10
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
    
    default_mean_accuracy_xgb = 0.0
    try:
        cv_scores_xgb = cross_val_score(model_pipeline_xgb_default, X_all, y_all, cv=skf, scoring='accuracy', n_jobs=1)
        
        print("\n--- 10-折交叉验证结果 (XGBoost 默认参数) ---")
        print(f"10 次试验的准确率分数: \n{np.round(cv_scores_xgb, 4)}")
        print("\n[ XGBoost 默认模型可信得分 ]")
        default_mean_accuracy_xgb = np.mean(cv_scores_xgb)
        print(f"平均准确率: {default_mean_accuracy_xgb * 100:.2f}%")
        print(f"标准差: +/- {np.std(cv_scores_xgb) * 100:.2f}%")
        
        print("\n正在生成交叉验证得分箱形图...")
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=cv_scores_xgb, orient='h', palette='Oranges')
        sns.stripplot(data=cv_scores_xgb, orient='h', color='black', alpha=0.7, jitter=0.1)
        plt.axvline(np.mean(cv_scores_xgb), color='blue', linestyle='--', label=f'平均值: {np.mean(cv_scores_xgb):.4f}')
        plt.title(f'10-折交叉验证得分分布 (SMOTE + 默认 XGBoost)')
        plt.xlabel('准确率 (Accuracy)')
        plt.xlim(0.8, 1.0)
        plt.legend()
        plt.savefig("cv_scores_boxplot_xgb.png")
        print(f"图表已保存为: cv_scores_boxplot_xgb.png")
        plt.close()

    except Exception as e_cv:
        print(f"交叉验证执行失败: {e_cv}")
    
    # --- 13. 网格搜索 (Grid Search CV) for XGBoost ---
    print("\n步骤 13/13: 正在执行网格搜索 (Grid Search CV) 来寻找 XGBoost 最佳参数... (这会 *非常* 耗时, 可能需要 10-20 分钟)")

    # 定义 XGBoost 的参数网格
    param_grid_xgb = {
        'model__n_estimators': [100, 150],       # 树的数量
        'model__max_depth': [5, 10, 15],         # 树的深度
        'model__learning_rate': [0.1, 0.05]      # 学习率
    }
    # 总共的组合数: 2 * 3 * 2 = 12 种
    # 交叉验证折数: 5
    # 总共的训练次数: 12 * 5 = 60 次
    
    pipeline_for_gridsearch_xgb = ImbPipeline([
        ('smote', SMOTE(random_state=42, n_jobs=-1)),
        ('model', xgb.XGBClassifier(random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss'))
    ])

    grid_search_xgb = GridSearchCV(estimator=pipeline_for_gridsearch_xgb,
                                   param_grid=param_grid_xgb,
                                   cv=5, 
                                   scoring='accuracy',
                                   n_jobs=1,
                                   verbose=2) # 打印详细进度

    try:
        grid_search_xgb.fit(X_all, y_all)
        
        print("\n--- 网格搜索 (Grid Search) 结果 (XGBoost) ---")
        best_accuracy_xgb = grid_search_xgb.best_score_
        print(f"\n[ 最佳得分 ]")
        print(f"找到的最佳平均准确率 (来自 5-折 CV): {best_accuracy_xgb * 100:.2f}%")
        if default_mean_accuracy_xgb > 0:
            print(f"(对比: 默认 XGBoost 的 10-折 CV 平均准确率: {default_mean_accuracy_xgb * 100:.2f}%)")
        
        print("\n[ 最佳参数组合 ]")
        print("电脑找到的最佳参数设置是:")
        print(grid_search_xgb.best_params_)
        
        print("\n[ 性能提升 ]")
        if default_mean_accuracy_xgb > 0 and best_accuracy_xgb > default_mean_accuracy_xgb:
            print(f"通过调优, XGBoost 性能从 {default_mean_accuracy_xgb * 100:.2f}% 提升到了 {best_accuracy_xgb * 100:.2f}%。")
        elif default_mean_accuracy_xgb > 0:
            print(f"调优后的得分 {best_accuracy_xgb * 100:.2f}% 未能超越默认参数的 {default_mean_accuracy_xgb * 100:.2f}%。")
            print("这说明 XGBoost 的默认参数已经非常接近最优解了。")
        else:
            print("已找到最佳参数。")

    except Exception as e_grid:
        print(f"网格搜索执行失败: {e_grid}")
    
    end_time_total = time.time()
    print(f"\n--- 完整流程结束 (总耗时: {(end_time_total - start_time_total) / 60:.2f} 分钟) ---")

except FileNotFoundError:
    print("\n--- 错误: 文件未找到 ---")
    print("找不到 'train_processed.csv' 或 'test_processed.csv'。")
except ImportError:
    print(f"\n--- 错误: 缺少库 ---")
    print("请先安装 'xgboost', 'imbalanced-learn', 'matplotlib' 和 'seaborn' 库。")
    print("您可以在 notebook 的一个单元格中运行: !pip install xgboost imbalanced-learn matplotlib seaborn")
except Exception as e:
    print(f"\n--- 处理过程中发生错误 ---")
    print(f"错误详情: {e}")


中文字体 'SimHei' 加载成功。
--- 机器学习“王牌”模型 (XGBoost 完整分析) ---
步骤 1/13: 正在加载 'train_processed.csv' 和 'test_processed.csv'...
数据加载完毕。
步骤 2/13: 正在分离特征 (X) 和目标 (y)...
全部数据: (1341, 46)

步骤 3/13: 正在应用 SMOTE 来平衡 *单次* 训练数据...
SMOTE 处理后训练集形状: (1857, 46)

步骤 4/13: 正在使用 (SMOTE) 平衡后的数据训练 *单次* XGBoost 模型...
模型训练完成。

步骤 5/13: 正在使用 *原始* 测试集评估 (XGBoost) 模型...

--- 模型评估结果 (SMOTE + XGBoost) [单次 80/20 拆分] ---

[ 1. 总体准确率 ]
模型 (XGBoost) 准确率为: 91.08%

[ 2. 详细分类报告 ]
                  precision    recall  f1-score   support

A-type (Class 0)       0.97      0.92      0.94       155
I-type (Class 1)       0.87      0.90      0.88        81
S-type (Class 2)       0.79      0.91      0.85        33

        accuracy                           0.91       269
       macro avg       0.87      0.91      0.89       269
    weighted avg       0.92      0.91      0.91       269


步骤 7/13: 正在生成混淆矩阵图表...
图表已保存为: cm_xgboost.png

步骤 8/13: 正在生成分类报告柱状图...
图表已保存为: report_bar_chart_xgb.png

步骤 9/13: 正在生成特征重要性柱状图 (XGBoost 模型)...
图表已保存为

In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb # <-- 导入 XGBoost
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import validation_curve
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
import time # <-- 导入 time 模块来计时
# --- 新增: 导入预处理模块 ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer
# --- 导入结束 ---

# 忽略将来可能的警告
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 设置 Matplotlib 支持中文 ---
try:
    plt.rcParams['font.sans-serif'] = ['SimHei']
    print("中文字体 'SimHei' 加载成功。")
except Exception as e:
    print(f"中文字体 'SimHei' 设置失败: {e}")
plt.rcParams['axes.unicode_minus'] = False
# --- 设置结束 ---


print("--- 机器学习“王牌”模型 (XGBoost 完整分析) ---")
start_time_total = time.time() # 记录总开始时间

try:
    # --- 步骤 1: 加载 *原始* Excel 文件并预处理 ---
    print("步骤 1/14: 正在加载 *原始* Excel 文件 'ASI分类.xls'...")
    try:
        df_raw = pd.read_excel("ASI分类.xls", sheet_name="Sheet1", header=1)
    except FileNotFoundError:
        print("错误: 找不到 'ASI分类.xls'。请确保它和您的 Jupyter 笔记本在同一个文件夹中。")
        raise
    except ImportError:
        print("错误: 缺少 'xlrd' 库。请运行: !pip install xlrd")
        raise
        
    print("步骤 2/14: 正在清理和预处理数据...")
    # 2a. 清理数字列
    cols_to_convert = []
    try:
        start_col_index = df_raw.columns.get_loc('SiO2')
        end_col_col = 'Cs ' if 'Cs ' in df_raw.columns else 'Cs'
        if end_col_col not in df_raw.columns:
             end_col_index = df_raw.columns.get_loc('U')
        else:
             end_col_index = df_raw.columns.get_loc(end_col_col)
        cols_to_convert = df_raw.columns[start_col_index : end_col_index + 1]
    except KeyError: pass
    for col in cols_to_convert:
        if df_raw[col].dtype == 'object':
            df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

    # 2b. 定义 X (原始特征) 和 y (目标)
    cols_to_drop_initial = ['No.', 'Type', 'Type-1']
    cols_present = [col for col in cols_to_drop_initial if col in df_raw.columns]
    X_raw = df_raw.drop(columns=cols_present)
    y_raw = df_raw['Type']
    
    # 2c. 剔除高缺失列
    missing_percentage = X_raw.isnull().mean()
    cols_to_drop_missing = missing_percentage[missing_percentage > 0.4].index
    X_raw = X_raw.drop(columns=cols_to_drop_missing)
    feature_names = X_raw.columns # 保存特征名称
    
    # 2d. 编码和清理
    le = LabelEncoder()
    y_clean = y_raw.dropna()
    y_indices = y_clean.index
    X_clean = X_raw.loc[y_indices] # X_clean 是带 NaN 的 Pandas DataFrame
    y_encoded = le.fit_transform(y_clean) # y_encoded 是 NumPy 数组
    class_names_plot = list(le.classes_) # ['A-type', 'I-type', 'S-type']
    
    # 2e. 拆分 (注意: X_train_raw 和 X_test_raw 保留了原始索引)
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_clean, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )

    # 2f. 插补和缩放 (用于单次评估 和 *完整数据*)
    imputer = KNNImputer(n_neighbors=5)
    scaler = StandardScaler()
    
    # (用于单次评估)
    X_train_processed = imputer.fit_transform(X_train_raw)
    X_train_processed = scaler.fit_transform(X_train_processed)
    
    X_test_processed = imputer.transform(X_test_raw)
    X_test_processed = scaler.transform(X_test_processed)
    
    # (用于 CV, GridSearch, Validation Curve)
    # 我们需要在 X_clean (完整数据) 上插补和缩放, 得到一个完整的处理后数组
    X_all_processed = imputer.fit_transform(X_clean)
    X_all_processed = scaler.fit_transform(X_all_processed)
    y_all = y_encoded # y_all 现在是 y_encoded (NumPy 数组)
    
    print("数据预处理完毕。")
    print(f"训练集: {X_train_processed.shape}, 测试集: {X_test_processed.shape}, 全部数据 (处理后): {X_all_processed.shape}")
    # --- 预处理结束 ---

    # 3. SMOTE (用于单次训练)
    print("\n步骤 3/14: 正在应用 SMOTE 来平衡 *单次* 训练数据...")
    smote = SMOTE(random_state=42, n_jobs=-1)
    X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train) # 使用处理后的数据
    print(f"SMOTE 处理后训练集形状: {X_train_smote.shape}")

    # 4. 训练 XGBoost (单次)
    print("\n步骤 4/14: 正在使用 (SMOTE) 平衡后的数据训练 *单次* XGBoost 模型...")
    xgb_model_single = xgb.XGBClassifier(
        n_estimators=100, 
        random_state=42, 
        n_jobs=-1, 
        use_label_encoder=False, 
        eval_metric='mlogloss'
    )
    xgb_model_single.fit(X_train_smote, y_train_smote)
    print("模型训练完成。")

    # 5. 评估 XGBoost (单次)
    print("\n步骤 5/14: 正在使用 *原始* 测试集评估 (XGBoost) 模型...")
    y_pred_xgb = xgb_model_single.predict(X_test_processed) # 使用处理后的数据
    accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
    print(f"\n--- 模型评估结果 (SMOTE + XGBoost) [单次 80/20 拆分] ---")
    print(f"\n[ 1. 总体准确率 ]")
    print(f"模型 (XGBoost) 准确率为: {accuracy_xgb * 100:.2f}%")
    
    # 6. 分类报告 (单次)
    print("\n[ 2. 详细分类报告 ]")
    target_names_report = ['A-type (Class 0)', 'I-type (Class 1)', 'S-type (Class 2)']
    report_xgb = classification_report(y_test, y_pred_xgb, target_names=target_names_report)
    print(report_xgb)

    # 7. 混淆矩阵 (单次)
    print("\n步骤 7/14: 正在生成混淆矩阵图表...")
    cm_xgb = confusion_matrix(y_test, y_pred_xgb)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='YlGnBu',
                xticklabels=class_names_plot, yticklabels=class_names_plot)
    plt.title(f'SMOTE + XGBoost 混淆矩阵\n(总体准确率: {accuracy_xgb * 100:.2f}%)')
    plt.ylabel('真实类别 (True Label)')
    plt.xlabel('预测类别 (Predicted Label)')
    plt.savefig("cm_xgboost.png")
    print(f"图表已保存为: cm_xgboost.png")
    plt.close()

    # 8. 分类报告柱状图 (单次)
    print("\n步骤 8/14: 正在生成分类报告柱状图...")
    report_dict_xgb = classification_report(y_test, y_pred_xgb, target_names=class_names_plot, output_dict=True)
    report_df_xgb = pd.DataFrame(report_dict_xgb).transpose().loc[class_names_plot, ['precision', 'recall', 'f1-score']]
    report_df_xgb.plot(kind='bar', figsize=(12, 7), rot=0, colormap='summer')
    plt.title('SMOTE + XGBoost 分类报告指标')
    plt.ylabel('得分 (Score)')
    plt.xlabel('岩石类别')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.ylim(0, 1.05)
    for p in plt.gca().patches:
        plt.gca().annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='center', xytext=(0, 9), textcoords='offset points')
    plt.savefig("report_bar_chart_xgb.png")
    print(f"图表已保存为: report_bar_chart_xgb.png")
    plt.close()

    # 9. 特征重要性柱状图 (单次)
    print("\n步骤 9/14: 正在生成特征重要性柱状图 (XGBoost 模型)...")
    importances_xgb = xgb_model_single.feature_importances_
    feature_importance_xgb_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances_xgb
    }).sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(12, 9))
    sns.barplot(x='Importance', y='Feature', data=feature_importance_xgb_df.head(15), palette='autumn')
    plt.title('SMOTE + XGBoost - Top 15 重要特征')
    plt.xlabel('重要性得分 (F-Score / Gain)')
    plt.ylabel('特征 (Feature)')
    plt.savefig("feature_importance_xgb.png")
    print(f"图表已保存为: feature_importance_xgb.png")
    plt.close()

    # 10. 特征散点图 (单次)
    print("\n步骤 10/14: 正在生成最重要特征散点图 (XGBoost 模型)...")
    top_feature_1_xgb = feature_importance_xgb_df.iloc[0]['Feature']
    top_feature_2_xgb = feature_importance_xgb_df.iloc[1]['Feature']
    y_all_named_scatter = y_clean.map({0: 'A-type', 1: 'I-type', 2: 'S-type'})
    scatter_df_xgb = pd.DataFrame({'Feature_1': X_clean[top_feature_1_xgb], 'Feature_2': X_clean[top_feature_2_xgb], 'Type': y_all_named_scatter})
    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=scatter_df_xgb, x='Feature_1', y='Feature_2', hue='Type', alpha=0.7, palette='dark')
    plt.title(f'XGBoost 模型 最重要特征散点图\n({top_feature_1_xgb} vs {top_feature_2_xgb})')
    plt.xlabel(top_feature_1_xgb)
    plt.ylabel(top_feature_2_xgb)
    plt.savefig("scatter_top_features_xgb.png")
    print(f"图表已保存为: scatter_top_features_xgb.png")
    plt.close()

    # 11. 验证曲线 (折线图)
    print("\n步骤 11/14: 正在生成 XGBoost 验证曲线(折线图)... (这可能需要一点时间)")
    try:
        param_range = [1, 3, 5, 7, 10]
        train_scores, test_scores = validation_curve(
            xgb.XGBClassifier(n_estimators=50, random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss'),
            X_all_processed, y_all, # X_all_processed 是处理后的 NumPy 数组
            param_name="max_depth",
            param_range=param_range,
            cv=3,
            scoring="accuracy",
            n_jobs=1 
        )
        train_scores_mean = np.mean(train_scores, axis=1)
        test_scores_mean = np.mean(test_scores, axis=1)
        plt.figure(figsize=(10, 6))
        plt.title("XGBoost 验证曲线 (参数 'max_depth')")
        plt.xlabel("树的最大深度 (max_depth)")
        plt.ylabel("准确率 (Accuracy)")
        plt.ylim(0.0, 1.1)
        plt.plot(param_range, train_scores_mean, label="训练得分", color="r", lw=2)
        plt.plot(param_range, test_scores_mean, label="交叉验证得分 (测试)", color="g", lw=2)
        plt.legend(loc="best")
        plt.savefig("validation_curve_xgb.png")
        print(f"图表已保存为: validation_curve_xgb.png")
        plt.close()

    except Exception as e_val:
        print(f"验证曲线绘制失败: {e_val}")
    
    # --- 12. 10-折交叉验证 (XGBoost 默认参数) ---
    print("\n步骤 12/14: 正在执行 10-折交叉验证 (XGBoost 默认参数)... (这会需要几分钟时间)")
    
    # 【*** 关键修复 1 ***】
    # 创建一个 *只包含 SMOTE 和 模型* 的流水线
    # 预处理(插补/缩放)将在流水线 *外部* 完成
    model_pipeline_xgb_default = ImbPipeline([
        ('smote', SMOTE(random_state=42, n_jobs=-1)),
        ('model', xgb.XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss'))
    ])
    
    k_folds = 10
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
    
    default_mean_accuracy_xgb = 0.0
    try:
        # 【*** 关键修复 2 ***】
        # 在 X_all_processed (无NaN) 和 y_all (NumPy) 上运行
        cv_scores_xgb = cross_val_score(model_pipeline_xgb_default, X_all_processed, y_all, cv=skf, scoring='accuracy', n_jobs=1)
        
        print("\n--- 10-折交叉验证结果 (XGBoost 默认参数) ---")
        print(f"10 次试验的准确率分数: \n{np.round(cv_scores_xgb, 4)}")
        print("\n[ XGBoost 默认模型可信得分 ]")
        default_mean_accuracy_xgb = np.mean(cv_scores_xgb)
        print(f"平均准确率: {default_mean_accuracy_xgb * 100:.2f}%")
        print(f"标准差: +/- {np.std(cv_scores_xgb) * 100:.2f}%")
        
        print("\n正在生成交叉验证得分箱形图...")
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=cv_scores_xgb, orient='h', palette='Oranges')
        sns.stripplot(data=cv_scores_xgb, orient='h', color='black', alpha=0.7, jitter=0.1)
        plt.axvline(np.mean(cv_scores_xgb), color='blue', linestyle='--', label=f'平均值: {np.mean(cv_scores_xgb):.4f}')
        plt.title(f'10-折交叉验证得分分布 (SMOTE + 默认 XGBoost)')
        plt.xlabel('准确率 (Accuracy)')
        plt.xlim(0.8, 1.0)
        plt.legend()
        plt.savefig("cv_scores_boxplot_xgb.png")
        print(f"图表已保存为: cv_scores_boxplot_xgb.png")
        plt.close()

    except Exception as e_cv:
        print(f"交叉验证执行失败: {e_cv}")
    
    # --- 13. 网格搜索 (Grid Search CV) for XGBoost ---
    print("\n步骤 13/14: 正在执行网格搜索 (Grid Search CV) 来寻找 XGBoost 最佳参数... (这会 *非常* 耗时, 可能需要 10-20 分钟)")

    param_grid_xgb = {
        'model__n_estimators': [100, 150],
        'model__max_depth': [5, 10, 15],
        'model__learning_rate': [0.1, 0.05]
    }
    
    # 【*** 关键修复 3 ***】
    # 创建一个 *只包含 SMOTE 和 模型* 的流水线
    pipeline_for_gridsearch_xgb = ImbPipeline([
        ('smote', SMOTE(random_state=42, n_jobs=-1)),
        ('model', xgb.XGBClassifier(random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss'))
    ])

    grid_search_xgb = GridSearchCV(estimator=pipeline_for_gridsearch_xgb,
                                   param_grid=param_grid_xgb,
                                   cv=5, 
                                   scoring='accuracy',
                                   n_jobs=1,
                                   verbose=2)

    try:
        # 【*** 关键修复 4 ***】
        # 在 X_all_processed (无NaN) 和 y_all (NumPy) 上运行
        grid_search_xgb.fit(X_all_processed, y_all)
        
        print("\n--- 网格搜索 (Grid Search) 结果 (XGBoost) ---")
        best_accuracy_xgb = grid_search_xgb.best_score_
        print(f"\n[ 最佳得分 ]")
        print(f"找到的最佳平均准确率 (来自 5-折 CV): {best_accuracy_xgb * 100:.2f}%")
        if default_mean_accuracy_xgb > 0:
            print(f"(对比: 默认 XGBoost 的 10-折 CV 平均准确率: {default_mean_accuracy_xgb * 100:.2f}%)")
        
        print("\n[ 最佳参数组合 ]")
        print("电脑找到的最佳参数设置是:")
        print(grid_search_xgb.best_params_)
        
        print("\n[ 性能提升 ]")
        if default_mean_accuracy_xgb > 0 and best_accuracy_xgb > default_mean_accuracy_xgb:
            print(f"通过调优, XGBoost 性能从 {default_mean_accuracy_xgb * 100:.2f}% 提升到了 {best_accuracy_xgb * 100:.2f}%。")
        elif default_mean_accuracy_xgb > 0:
            print(f"调优后的得分 {best_accuracy_xgb * 100:.2f}% 未能超越默认参数的 {default_mean_accuracy_xgb * 100:.2f}%。")
            print("这说明 XGBoost 的默认参数已经非常接近最优解了。")
        else:
            print("已找到最佳参数。")

    except Exception as e_grid:
        print(f"网格搜索执行失败: {e_grid}")
    
    # --- 14. 【*** 深入错误分析 ***】 ---
    print(f"\n步骤 14/14: 正在执行深入错误分析 (基于步骤 5 的 {accuracy_xgb*100:.2f}% 模型)...")
    
    try:
        # 1. 找到在 X_test_raw 中被错判的样本的 *原始索引*
        error_mask = y_pred_xgb != y_test
        # X_test_raw 是一个保留了原始索引的 Pandas DataFrame
        error_indices = X_test_raw[error_mask].index 
        
        print(f"模型在测试集上总共错判了 {len(error_indices)} 个样本。")
        
        # 2. 从 *原始* df_raw 中提取这些行
        misclassified_samples_raw = df_raw.loc[error_indices]
        
        # 3. 准备用于分析的 DataFrame
        predictions_for_errors = y_pred_xgb[error_mask]
        true_labels_for_errors = y_test[error_mask]
        
        # 4. 将 0,1,2 转换回 A,I,S
        true_labels_named = le.inverse_transform(true_labels_for_errors)
        pred_labels_named = le.inverse_transform(predictions_for_errors)
        
        analysis_df = misclassified_samples_raw.copy()
        analysis_df['True_Type'] = true_labels_named
        analysis_df['Predicted_Type'] = pred_labels_named
        
        # 5. 定义我们关心的地化指标
        key_geochem_cols = ['True_Type', 'Predicted_Type', 'A/CNK', 'Zr+Nb+Ce+Y', 'Zr', 'Nb', 'Y', 'Hf', 'REE', 'Sr/Y']
        # 确保这些列都存在
        existing_key_columns = [col for col in key_geochem_cols if col in analysis_df.columns]
        
        print("\n--- 所有错判样本的“关键”原始化学成分 ---")
        # .to_string() 保证打印所有行
        print(analysis_df[existing_key_columns].to_string())
        
        # 6. 执行您建议的具体分析
        print("\n--- 深入分析 1: 错判为 'I-type' 的 'S-type' (S-type 特征: A/CNK > 1.1) ---")
        s_as_i = analysis_df[
            (analysis_df['True_Type'] == 'S-type') & 
            (analysis_df['Predicted_Type'] == 'I-type')
        ]
        if not s_as_i.empty:
            print(s_as_i[existing_key_columns])
            print(f"分析: {len(s_as_i)} 个 S-type 被错判为 I-type。请检查它们的 'A/CNK' 值。")
            print(f"   它们的 A/CNK 均值: {s_as_i['A/CNK'].mean():.3f} (S-type 通常 > 1.1)")
        else:
            print("模型没有将 S-type 错判为 I-type。")

        print("\n--- 深入分析 2: 错判为 'A-type' 的 'I-type' (A-type 特征: Zr+Nb+Ce+Y 高) ---")
        i_as_a = analysis_df[
            (analysis_df['True_Type'] == 'I-type') & 
            (analysis_df['Predicted_Type'] == 'A-type')
        ]
        if not i_as_a.empty:
            print(i_as_a[existing_key_columns])
            print(f"分析: {len(i_as_a)} 个 I-type 被错判为 A-type。请检查它们的高场强元素。")
            print(f"   它们的 'Zr+Nb+Ce+Y' 均值: {i_as_a['Zr+Nb+Ce+Y'].mean():.2f}")
        else:
            print("模型没有将 I-type 错判为 A-type。")

        print("\n--- 深入分析 3: 被错判的 'A-type' ---")
        a_as_other = analysis_df[
            (analysis_df['True_Type'] == 'A-type')
        ]
        if not a_as_other.empty:
            print(a_as_other[existing_key_columns])
            print(f"分析: {len(a_as_other)} 个 A-type 被错判。请检查它们的高场强元素。")
            print(f"   它们的 'Zr+Nb+Ce+Y' 均值: {a_as_other['Zr+Nb+Ce+Y'].mean():.2f} (A-type 通常很高)")
        else:
            print("模型没有错判 A-type。")
            
    except Exception as e_error:
        print(f"错误分析执行失败: {e_error}")
    # --- 错误分析结束 ---

    end_time_total = time.time()
    print(f"\n--- 完整流程结束 (总耗时: {(end_time_total - start_time_total) / 60:.2f} 分钟) ---")

except FileNotFoundError:
    print("\n--- 错误: 文件未找到 ---")
    print("找不到 'ASI分类.xls'。请确保它和您的 Jupyter 笔记本在同一个文件夹中。")
except ImportError:
    print(f"\n--- 错误: 缺少库 ---")
    print("请先安装 'xgboost', 'imbalanced-learn', 'matplotlib', 'seaborn', 'xlrd', 'openpyxl' 库。")
except Exception as e:
    print(f"\n--- 处理过程中发生错误 ---")
    print(f"错误详情: {e}")



中文字体 'SimHei' 加载成功。
--- 机器学习“王牌”模型 (XGBoost 完整分析) ---
步骤 1/14: 正在加载 *原始* Excel 文件 'ASI分类.xls'...
步骤 2/14: 正在清理和预处理数据...
数据预处理完毕。
训练集: (1072, 46), 测试集: (269, 46), 全部数据 (处理后): (1341, 46)

步骤 3/14: 正在应用 SMOTE 来平衡 *单次* 训练数据...
SMOTE 处理后训练集形状: (1857, 46)

步骤 4/14: 正在使用 (SMOTE) 平衡后的数据训练 *单次* XGBoost 模型...
模型训练完成。

步骤 5/14: 正在使用 *原始* 测试集评估 (XGBoost) 模型...

--- 模型评估结果 (SMOTE + XGBoost) [单次 80/20 拆分] ---

[ 1. 总体准确率 ]
模型 (XGBoost) 准确率为: 91.08%

[ 2. 详细分类报告 ]
                  precision    recall  f1-score   support

A-type (Class 0)       0.97      0.92      0.94       155
I-type (Class 1)       0.87      0.90      0.88        81
S-type (Class 2)       0.79      0.91      0.85        33

        accuracy                           0.91       269
       macro avg       0.87      0.91      0.89       269
    weighted avg       0.92      0.91      0.91       269


步骤 7/14: 正在生成混淆矩阵图表...
图表已保存为: cm_xgboost.png

步骤 8/14: 正在生成分类报告柱状图...
图表已保存为: report_bar_chart_xgb.png

步骤 9/14: 正在生成特征重要性柱状图 (XGBoost 

E:\anaconda3\envs\rain_pytorch\lib\site-packages\ipykernel_launcher.py:195: UserWarning: Ignoring `palette` because no `hue` variable has been assigned.


图表已保存为: validation_curve_xgb.png

步骤 12/14: 正在执行 10-折交叉验证 (XGBoost 默认参数)... (这会需要几分钟时间)

--- 10-折交叉验证结果 (XGBoost 默认参数) ---
10 次试验的准确率分数: 
[0.9111 0.9179 0.8955 0.9254 0.9179 0.8955 0.9403 0.8955 0.9552 0.903 ]

[ XGBoost 默认模型可信得分 ]
平均准确率: 91.57%
标准差: +/- 1.92%

正在生成交叉验证得分箱形图...
图表已保存为: cv_scores_boxplot_xgb.png

步骤 13/14: 正在执行网格搜索 (Grid Search CV) 来寻找 XGBoost 最佳参数... (这会 *非常* 耗时, 可能需要 10-20 分钟)
Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END model__learning_rate=0.1, model__max_depth=5, model__n_estimators=100; total time=   0.3s
[CV] END model__learning_rate=0.1, model__max_depth=5, model__n_estimators=100; total time=   0.3s
[CV] END model__learning_rate=0.1, model__max_depth=5, model__n_estimators=100; total time=   0.4s
[CV] END model__learning_rate=0.1, model__max_depth=5, model__n_estimators=100; total time=   0.3s
[CV] END model__learning_rate=0.1, model__max_depth=5, model__n_estimators=100; total time=   0.3s
[CV] END model__learning_rate=0.1, model__max_

In [5]:
import pandas as pd
import numpy as np
import xgboost as xgb # <-- 导入 XGBoost
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import validation_curve
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
import time # <-- 导入 time 模块来计时
# --- 新增: 导入预处理模块 ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer
# --- 导入结束 ---

# 忽略将来可能的警告
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 设置 Matplotlib 支持中文 ---
try:
    plt.rcParams['font.sans-serif'] = ['SimHei']
    print("中文字体 'SimHei' 加载成功。")
except Exception as e:
    print(f"中文字体 'SimHei' 设置失败: {e}")
plt.rcParams['axes.unicode_minus'] = False
# --- 设置结束 ---


print("--- 机器学习“王牌”模型 (XGBoost 完整分析) ---")
start_time_total = time.time() # 记录总开始时间

try:
    # --- 步骤 1: 加载 *原始* Excel 文件并预处理 ---
    print("步骤 1/15: 正在加载 *原始* Excel 文件 'ASI分类.xls'...")
    try:
        df_raw = pd.read_excel("ASI分类.xls", sheet_name="Sheet1", header=1)
    except FileNotFoundError:
        print("错误: 找不到 'ASI分类.xls'。请确保它和您的 Jupyter 笔记本在同一个文件夹中。")
        raise
    except ImportError:
        print("错误: 缺少 'xlrd' 库。请运行: !pip install xlrd")
        raise
        
    print("步骤 2/15: 正在清理和预处理数据...")
    # 2a. 清理数字列
    cols_to_convert = []
    try:
        start_col_index = df_raw.columns.get_loc('SiO2')
        end_col_col = 'Cs ' if 'Cs ' in df_raw.columns else 'Cs'
        if end_col_col not in df_raw.columns:
             end_col_index = df_raw.columns.get_loc('U')
        else:
             end_col_index = df_raw.columns.get_loc(end_col_col)
        cols_to_convert = df_raw.columns[start_col_index : end_col_index + 1]
    except KeyError: pass
    for col in cols_to_convert:
        if df_raw[col].dtype == 'object':
            df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

    # 2b. 定义 X (原始特征) 和 y (目标)
    cols_to_drop_initial = ['No.', 'Type', 'Type-1']
    cols_present = [col for col in cols_to_drop_initial if col in df_raw.columns]
    X_raw = df_raw.drop(columns=cols_present)
    y_raw = df_raw['Type']
    
    # 2c. 剔除高缺失列
    missing_percentage = X_raw.isnull().mean()
    cols_to_drop_missing = missing_percentage[missing_percentage > 0.4].index
    X_raw = X_raw.drop(columns=cols_to_drop_missing)
    # feature_names = X_raw.columns # 稍后在特征工程后更新
    
    # 2d. 编码和清理
    le = LabelEncoder()
    y_clean = y_raw.dropna()
    y_indices = y_clean.index
    X_clean = X_raw.loc[y_indices] # X_clean 是带 NaN 的 Pandas DataFrame
    y_encoded = le.fit_transform(y_clean) # y_encoded 是 NumPy 数组
    class_names_plot = list(le.classes_) # ['A-type', 'I-type', 'S-type']
    
    # --- 步骤 2.5/15: 【*** 全新步骤: 特征工程 ***】 ---
    print("\n步骤 2.5/15: 正在执行特征工程 (创建 'ACNK_Boundary_Risk')...")
    
    # 定义边界值
    boundary_value = 1.1
    
    # 计算到边界的距离 (处理 NaN)
    # .abs() 取绝对值
    distance = (X_clean['A/CNK'] - boundary_value).abs()
    
    # 处理距离为 0 的情况 (A/CNK == 1.1), 避免除以零 (np.inf)
    # 用一个很小的值 (e.g., 0.0001) 来代替 0
    distance = distance.replace(0, 1e-4) 
    
    # 计算风险特征 (1 / 距离)
    # A/CNK 本身为 NaN 的地方, distance 也会是 NaN, 
    # X_clean['ACNK_Boundary_Risk'] 也会是 NaN.
    # 这没问题, 因为 KNNImputer 会在下一步 (2f) 中自动插补这些 NaN.
    X_clean['ACNK_Boundary_Risk'] = 1 / distance
    
    # 确保 feature_names 列表也更新了
    feature_names = X_clean.columns
    print(f"特征工程完毕。新特征 'ACNK_Boundary_Risk' 已添加。")
    print(f"X_clean (带新特征) 的形状: {X_clean.shape}")
    # --- 特征工程结束 ---
    
    # 2e. 拆分 (注意: X_train_raw 和 X_test_raw 保留了原始索引)
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_clean, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )

    # 2f. 插补和缩放 (用于单次评估 和 *完整数据*)
    imputer = KNNImputer(n_neighbors=5)
    scaler = StandardScaler()
    
    # (用于单次评估)
    # X_train_processed 会自动包含 'ACNK_Boundary_Risk'
    X_train_processed = imputer.fit_transform(X_train_raw)
    X_train_processed = scaler.fit_transform(X_train_processed)
    
    X_test_processed = imputer.transform(X_test_raw)
    X_test_processed = scaler.transform(X_test_processed)
    
    # (用于 CV, GridSearch, Validation Curve)
    # X_all_processed 也会自动包含 'ACNK_Boundary_Risk'
    X_all_processed = imputer.fit_transform(X_clean)
    X_all_processed = scaler.fit_transform(X_all_processed)
    y_all = y_encoded # y_all 现在是 y_encoded (NumPy 数组)
    
    print("数据预处理完毕。")
    print(f"训练集: {X_train_processed.shape}, 测试集: {X_test_processed.shape}, 全部数据 (处理后): {X_all_processed.shape}")
    # --- 预处理结束 ---

    # 3. SMOTE (用于单次训练)
    print("\n步骤 3/15: 正在应用 SMOTE 来平衡 *单次* 训练数据...")
    smote = SMOTE(random_state=42, n_jobs=-1)
    X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train) # 使用处理后的数据
    print(f"SMOTE 处理后训练集形状: {X_train_smote.shape}")

    # 4. 训练 XGBoost (单次)
    print("\n步骤 4/15: 正在使用 (SMOTE) 平衡后的数据训练 *单次* XGBoost 模型...")
    xgb_model_single = xgb.XGBClassifier(
        n_estimators=100, 
        random_state=42, 
        n_jobs=-1, 
        use_label_encoder=False, 
        eval_metric='mlogloss'
    )
    xgb_model_single.fit(X_train_smote, y_train_smote)
    print("模型训练完成。")

    # 5. 评估 XGBoost (单次)
    print("\n步骤 5/15: 正在使用 *原始* 测试集评估 (XGBoost) 模型...")
    y_pred_xgb = xgb_model_single.predict(X_test_processed) # 使用处理后的数据
    accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
    print(f"\n--- 模型评估结果 (SMOTE + XGBoost) [单次 80/20 拆分] ---")
    print(f"\n[ 1. 总体准确率 ]")
    print(f"模型 (XGBoost) 准确率为: {accuracy_xgb * 100:.2f}%")
    
    # 6. 分类报告 (单次)
    print("\n[ 2. 详细分类报告 ]")
    target_names_report = ['A-type (Class 0)', 'I-type (Class 1)', 'S-type (Class 2)']
    report_xgb = classification_report(y_test, y_pred_xgb, target_names=target_names_report)
    print(report_xgb)

    # 7. 混淆矩阵 (单次)
    print("\n步骤 7/15: 正在生成混淆矩阵图表...")
    cm_xgb = confusion_matrix(y_test, y_pred_xgb)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='YlGnBu',
                xticklabels=class_names_plot, yticklabels=class_names_plot)
    plt.title(f'SMOTE + XGBoost 混淆矩阵\n(总体准确率: {accuracy_xgb * 100:.2f}%)')
    plt.ylabel('真实类别 (True Label)')
    plt.xlabel('预测类别 (Predicted Label)')
    plt.savefig("cm_xgboost.png")
    print(f"图表已保存为: cm_xgboost.png")
    plt.close()

    # 8. 分类报告柱状图 (单次)
    print("\n步骤 8/15: 正在生成分类报告柱状图...")
    report_dict_xgb = classification_report(y_test, y_pred_xgb, target_names=class_names_plot, output_dict=True)
    report_df_xgb = pd.DataFrame(report_dict_xgb).transpose().loc[class_names_plot, ['precision', 'recall', 'f1-score']]
    report_df_xgb.plot(kind='bar', figsize=(12, 7), rot=0, colormap='summer')
    plt.title('SMOTE + XGBoost 分类报告指标')
    plt.ylabel('得分 (Score)')
    plt.xlabel('岩石类别')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.ylim(0, 1.05)
    for p in plt.gca().patches:
        plt.gca().annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='center', xytext=(0, 9), textcoords='offset points')
    plt.savefig("report_bar_chart_xgb.png")
    print(f"图表已保存为: report_bar_chart_xgb.png")
    plt.close()

    # 9. 特征重要性柱状图 (单次)
    print("\n步骤 9/15: 正在生成特征重要性柱状图 (XGBoost 模型)...")
    importances_xgb = xgb_model_single.feature_importances_
    feature_importance_xgb_df = pd.DataFrame({
        'Feature': feature_names, # feature_names 现在包含了新特征
        'Importance': importances_xgb
    }).sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(12, 9))
    sns.barplot(x='Importance', y='Feature', data=feature_importance_xgb_df.head(15), palette='autumn')
    plt.title('SMOTE + XGBoost - Top 15 重要特征 (带新特征)')
    plt.xlabel('重要性得分 (F-Score / Gain)')
    plt.ylabel('特征 (Feature)')
    plt.savefig("feature_importance_xgb.png")
    print(f"图表已保存为: feature_importance_xgb.png")
    plt.close()

    # 10. 特征散点图 (单次)
    print("\n步骤 10/15: 正在生成最重要特征散点图 (XGBoost 模型)...")
    top_feature_1_xgb = feature_importance_xgb_df.iloc[0]['Feature']
    top_feature_2_xgb = feature_importance_xgb_df.iloc[1]['Feature']
    y_all_named_scatter = y_clean.map({0: 'A-type', 1: 'I-type', 2: 'S-type'})
    scatter_df_xgb = pd.DataFrame({'Feature_1': X_clean[top_feature_1_xgb], 'Feature_2': X_clean[top_feature_2_xgb], 'Type': y_all_named_scatter})
    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=scatter_df_xgb, x='Feature_1', y='Feature_2', hue='Type', alpha=0.7, palette='dark')
    plt.title(f'XGBoost 模型 最重要特征散点图\n({top_feature_1_xgb} vs {top_feature_2_xgb})')
    plt.xlabel(top_feature_1_xgb)
    plt.ylabel(top_feature_2_xgb)
    plt.savefig("scatter_top_features_xgb.png")
    print(f"图表已保存为: scatter_top_features_xgb.png")
    plt.close()

    # 11. 验证曲线 (折线图)
    print("\n步骤 11/15: 正在生成 XGBoost 验证曲线(折线图)... (这可能需要一点时间)")
    try:
        param_range = [1, 3, 5, 7, 10]
        train_scores, test_scores = validation_curve(
            xgb.XGBClassifier(n_estimators=50, random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss'),
            X_all_processed, y_all, # X_all_processed 是处理后的 NumPy 数组
            param_name="max_depth",
            param_range=param_range,
            cv=3,
            scoring="accuracy",
            n_jobs=1 
        )
        train_scores_mean = np.mean(train_scores, axis=1)
        test_scores_mean = np.mean(test_scores, axis=1)
        plt.figure(figsize=(10, 6))
        plt.title("XGBoost 验证曲线 (参数 'max_depth')")
        plt.xlabel("树的最大深度 (max_depth)")
        plt.ylabel("准确率 (Accuracy)")
        plt.ylim(0.0, 1.1)
        plt.plot(param_range, train_scores_mean, label="训练得分", color="r", lw=2)
        plt.plot(param_range, test_scores_mean, label="交叉验证得分 (测试)", color="g", lw=2)
        plt.legend(loc="best")
        plt.savefig("validation_curve_xgb.png")
        print(f"图表已保存为: validation_curve_xgb.png")
        plt.close()

    except Exception as e_val:
        print(f"验证曲线绘制失败: {e_val}")
    
    # --- 12. 10-折交叉验证 (XGBoost 默认参数) ---
    print("\n步骤 12/15: 正在执行 10-折交叉验证 (XGBoost 默认参数)... (这会需要几分钟时间)")
    
    # 流水线 *只包含* SMOTE 和 模型
    # 预处理(插补/缩放)在流水线 *外部* 完成
    model_pipeline_xgb_default = ImbPipeline([
        ('smote', SMOTE(random_state=42, n_jobs=-1)),
        ('model', xgb.XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss'))
    ])
    
    k_folds = 10
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
    
    default_mean_accuracy_xgb = 0.0
    try:
        # 在 X_all_processed (无NaN) 和 y_all (NumPy) 上运行
        cv_scores_xgb = cross_val_score(model_pipeline_xgb_default, X_all_processed, y_all, cv=skf, scoring='accuracy', n_jobs=1)
        
        print("\n--- 10-折交叉验证结果 (XGBoost 默认参数, 带新特征) ---")
        print(f"10 次试验的准确率分数: \n{np.round(cv_scores_xgb, 4)}")
        print("\n[ XGBoost 默认模型可信得分 ]")
        default_mean_accuracy_xgb = np.mean(cv_scores_xgb)
        print(f"平均准确率: {default_mean_accuracy_xgb * 100:.2f}%")
        print(f"标准差: +/- {np.std(cv_scores_xgb) * 100:.2f}%")
        print("(对比: 上一版无新特征的得分是 91.57%)")
        
        print("\n正在生成交叉验证得分箱形图...")
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=cv_scores_xgb, orient='h', palette='Oranges')
        sns.stripplot(data=cv_scores_xgb, orient='h', color='black', alpha=0.7, jitter=0.1)
        plt.axvline(np.mean(cv_scores_xgb), color='blue', linestyle='--', label=f'平均值: {np.mean(cv_scores_xgb):.4f}')
        plt.title(f'10-折交叉验证得分分布 (SMOTE + 默认 XGBoost, 带新特征)')
        plt.xlabel('准确率 (Accuracy)')
        plt.xlim(0.8, 1.0)
        plt.legend()
        plt.savefig("cv_scores_boxplot_xgb.png")
        print(f"图表已保存为: cv_scores_boxplot_xgb.png")
        plt.close()

    except Exception as e_cv:
        print(f"交叉验证执行失败: {e_cv}")
    
    # --- 13. 网格搜索 (Grid Search CV) for XGBoost ---
    print("\n步骤 13/15: 正在执行网格搜索 (Grid Search CV) 来寻找 XGBoost 最佳参数... (这会 *非常* 耗时, 可能需要 10-20 分钟)")

    param_grid_xgb = {
        'model__n_estimators': [100, 150],
        'model__max_depth': [5, 10, 15],
        'model__learning_rate': [0.1, 0.05]
    }
    
    # 流水线 *只包含* SMOTE 和 模型
    pipeline_for_gridsearch_xgb = ImbPipeline([
        ('smote', SMOTE(random_state=42, n_jobs=-1)),
        ('model', xgb.XGBClassifier(random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss'))
    ])

    grid_search_xgb = GridSearchCV(estimator=pipeline_for_gridsearch_xgb,
                                   param_grid=param_grid_xgb,
                                   cv=5, 
                                   scoring='accuracy',
                                   n_jobs=1,
                                   verbose=2)

    try:
        # 在 X_all_processed (无NaN) 和 y_all (NumPy) 上运行
        grid_search_xgb.fit(X_all_processed, y_all)
        
        print("\n--- 网格搜索 (Grid Search) 结果 (XGBoost, 带新特征) ---")
        best_accuracy_xgb = grid_search_xgb.best_score_
        print(f"\n[ 最佳得分 ]")
        print(f"找到的最佳平均准确率 (来自 5-折 CV): {best_accuracy_xgb * 100:.2f}%")
        if default_mean_accuracy_xgb > 0:
            print(f"(对比: 默认 XGBoost 的 10-折 CV 平均准确率: {default_mean_accuracy_xgb * 100:.2f}%)")
        
        print("\n[ 最佳参数组合 ]")
        print("电脑找到的最佳参数设置是:")
        print(grid_search_xgb.best_params_)
        
        print("\n[ 性能提升 ]")
        if default_mean_accuracy_xgb > 0 and best_accuracy_xgb > default_mean_accuracy_xgb:
            print(f"通过调优, XGBoost 性能从 {default_mean_accuracy_xgb * 100:.2f}% 提升到了 {best_accuracy_xgb * 100:.2f}%。")
        elif default_mean_accuracy_xgb > 0:
            print(f"调优后的得分 {best_accuracy_xgb * 100:.2f}% 未能超越默认参数的 {default_mean_accuracy_xgb * 100:.2f}%。")
            print("这说明 XGBoost 的默认参数已经非常接近最优解了。")
        else:
            print("已找到最佳参数。")

    except Exception as e_grid:
        print(f"网格搜索执行失败: {e_grid}")
    
    # --- 14. 【*** 深入错误分析 ***】 ---
    print(f"\n步骤 14/15: 正在执行深入错误分析 (基于步骤 5 的 {accuracy_xgb*100:.2f}% 模型)...")
    
    try:
        # 1. 找到在 X_test_raw 中被错判的样本的 *原始索引*
        error_mask = y_pred_xgb != y_test
        # X_test_raw 是一个保留了原始索引的 Pandas DataFrame
        error_indices = X_test_raw[error_mask].index 
        
        print(f"模型在测试集上总共错判了 {len(error_indices)} 个样本。")
        
        # 2. 从 *原始* df_raw 中提取这些行
        misclassified_samples_raw = df_raw.loc[error_indices]
        
        # 3. 准备用于分析的 DataFrame
        predictions_for_errors = y_pred_xgb[error_mask]
        true_labels_for_errors = y_test[error_mask]
        
        # 4. 将 0,1,2 转换回 A,I,S
        true_labels_named = le.inverse_transform(true_labels_for_errors)
        pred_labels_named = le.inverse_transform(predictions_for_errors)
        
        analysis_df = misclassified_samples_raw.copy()
        analysis_df['True_Type'] = true_labels_named
        analysis_df['Predicted_Type'] = pred_labels_named
        
        # 5. 定义我们关心的地化指标 (加入我们的新特征!)
        key_geochem_cols = ['True_Type', 'Predicted_Type', 'A/CNK', 'ACNK_Boundary_Risk', 'Zr+Nb+Ce+Y', 'Zr', 'Nb', 'Y', 'Hf']
        # 确保这些列都存在
        existing_key_columns = [col for col in key_geochem_cols if col in analysis_df.columns]
        
        print("\n--- 所有错判样本的“关键”原始化学成分 ---")
        # .to_string() 保证打印所有行
        print(analysis_df[existing_key_columns].to_string())
        
        # 6. 执行您建议的具体分析
        print("\n--- 深入分析 1: 错判为 'I-type' 的 'S-type' (S-type 特征: A/CNK > 1.1) ---")
        s_as_i = analysis_df[
            (analysis_df['True_Type'] == 'S-type') & 
            (analysis_df['Predicted_Type'] == 'I-type')
        ]
        if not s_as_i.empty:
            print(s_as_i[existing_key_columns])
            print(f"分析: {len(s_as_i)} 个 S-type 被错判为 I-type。请检查它们的 'A/CNK' 值。")
            print(f"   它们的 A/CNK 均值: {s_as_i['A/CNK'].mean():.3f} (S-type 通常 > 1.1)")
        else:
            print("模型没有将 S-type 错判为 I-type。")

        print("\n--- 深入分析 2: 错判为 'A-type' 的 'I-type' (A-type 特征: Zr+Nb+Ce+Y 高) ---")
        i_as_a = analysis_df[
            (analysis_df['True_Type'] == 'I-type') & 
            (analysis_df['Predicted_Type'] == 'A-type')
        ]
        if not i_as_a.empty:
            print(i_as_a[existing_key_columns])
            print(f"分析: {len(i_as_a)} 个 I-type 被错判为 A-type。请检查它们的高场强元素。")
            print(f"   它们的 'Zr+Nb+Ce+Y' 均值: {i_as_a['Zr+Nb+Ce+Y'].mean():.2f}")
        else:
            print("模型没有将 I-type 错判为 A-type。")

        print("\n--- 深入分析 3: 被错判的 'A-type' ---")
        a_as_other = analysis_df[
            (analysis_df['True_Type'] == 'A-type')
        ]
        if not a_as_other.empty:
            print(a_as_other[existing_key_columns])
            print(f"分析: {len(a_as_other)} 个 A-type 被错判。请检查它们的高场强元素。")
            print(f"   它们的 'Zr+Nb+Ce+Y' 均值: {a_as_other['Zr+Nb+Ce+Y'].mean():.2f} (A-type 通常很高)")
        else:
            print("模型没有错判 A-type。")
            
    except Exception as e_error:
        print(f"错误分析执行失败: {e_error}")
    # --- 错误分析结束 ---

    end_time_total = time.time()
    print(f"\n--- 完整流程结束 (总耗时: {(end_time_total - start_time_total) / 60:.2f} 分钟) ---")

except FileNotFoundError:
    print("\n--- 错误: 文件未找到 ---")
    print("找不到 'ASI分类.xls'。请确保它和您的 Jupyter 笔记本在同一个文件夹中。")
except ImportError:
    print(f"\n--- 错误: 缺少库 ---")
    print("请先安装 'xgboost', 'imbalanced-learn', 'matplotlib', 'seaborn', 'xlrd', 'openpyxl' 库。")
except Exception as e:
    print(f"\n--- 处理过程中发生错误 ---")
    print(f"错误详情: {e}")



中文字体 'SimHei' 加载成功。
--- 机器学习“王牌”模型 (XGBoost 完整分析) ---
步骤 1/15: 正在加载 *原始* Excel 文件 'ASI分类.xls'...
步骤 2/15: 正在清理和预处理数据...

步骤 2.5/15: 正在执行特征工程 (创建 'ACNK_Boundary_Risk')...
特征工程完毕。新特征 'ACNK_Boundary_Risk' 已添加。
X_clean (带新特征) 的形状: (1341, 47)
数据预处理完毕。
训练集: (1072, 47), 测试集: (269, 47), 全部数据 (处理后): (1341, 47)

步骤 3/15: 正在应用 SMOTE 来平衡 *单次* 训练数据...
SMOTE 处理后训练集形状: (1857, 47)

步骤 4/15: 正在使用 (SMOTE) 平衡后的数据训练 *单次* XGBoost 模型...
模型训练完成。

步骤 5/15: 正在使用 *原始* 测试集评估 (XGBoost) 模型...

--- 模型评估结果 (SMOTE + XGBoost) [单次 80/20 拆分] ---

[ 1. 总体准确率 ]
模型 (XGBoost) 准确率为: 91.08%

[ 2. 详细分类报告 ]
                  precision    recall  f1-score   support

A-type (Class 0)       0.96      0.92      0.94       155
I-type (Class 1)       0.87      0.89      0.88        81
S-type (Class 2)       0.82      0.94      0.87        33

        accuracy                           0.91       269
       macro avg       0.88      0.91      0.90       269
    weighted avg       0.91      0.91      0.91       269


步骤 7/15: 正在生成混淆矩阵图

E:\anaconda3\envs\rain_pytorch\lib\site-packages\ipykernel_launcher.py:222: UserWarning: Ignoring `palette` because no `hue` variable has been assigned.


图表已保存为: validation_curve_xgb.png

步骤 12/15: 正在执行 10-折交叉验证 (XGBoost 默认参数)... (这会需要几分钟时间)

--- 10-折交叉验证结果 (XGBoost 默认参数, 带新特征) ---
10 次试验的准确率分数: 
[0.9259 0.8955 0.903  0.9254 0.9179 0.9104 0.9552 0.9254 0.9552 0.8806]

[ XGBoost 默认模型可信得分 ]
平均准确率: 91.95%
标准差: +/- 2.26%
(对比: 上一版无新特征的得分是 91.57%)

正在生成交叉验证得分箱形图...
图表已保存为: cv_scores_boxplot_xgb.png

步骤 13/15: 正在执行网格搜索 (Grid Search CV) 来寻找 XGBoost 最佳参数... (这会 *非常* 耗时, 可能需要 10-20 分钟)
Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END model__learning_rate=0.1, model__max_depth=5, model__n_estimators=100; total time=   0.4s
[CV] END model__learning_rate=0.1, model__max_depth=5, model__n_estimators=100; total time=   0.4s
[CV] END model__learning_rate=0.1, model__max_depth=5, model__n_estimators=100; total time=   0.5s
[CV] END model__learning_rate=0.1, model__max_depth=5, model__n_estimators=100; total time=   0.4s
[CV] END model__learning_rate=0.1, model__max_depth=5, model__n_estimators=100; total time=   0.5s
[CV] END model_

In [6]:
import pandas as pd
import numpy as np
import xgboost as xgb # <-- 导入 XGBoost
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import validation_curve
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
import time # <-- 导入 time 模块来计时
# --- 新增: 导入预处理模块 ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer
# --- 导入结束 ---

# 忽略将来可能的警告
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 设置 Matplotlib 支持中文 ---
try:
    plt.rcParams['font.sans-serif'] = ['SimHei']
    print("中文字体 'SimHei' 加载成功。")
except Exception as e:
    print(f"中文字体 'SimHei' 设置失败: {e}")
plt.rcParams['axes.unicode_minus'] = False
# --- 设置结束 ---


print("--- 机器学习“王牌”模型 (XGBoost 完整分析) ---")
start_time_total = time.time() # 记录总开始时间

try:
    # --- 步骤 1: 加载 *原始* Excel 文件并预处理 ---
    print("步骤 1/16: 正在加载 *原始* Excel 文件 'ASI分类.xls'...")
    try:
        df_raw = pd.read_excel("ASI分类.xls", sheet_name="Sheet1", header=1)
    except FileNotFoundError:
        print("错误: 找不到 'ASI分类.xls'。请确保它和您的 Jupyter 笔记本在同一个文件夹中。")
        raise
    except ImportError:
        print("错误: 缺少 'xlrd' 库。请运行: !pip install xlrd")
        raise
        
    print("步骤 2/16: 正在清理和预处理数据...")
    # 2a. 清理数字列
    cols_to_convert = []
    try:
        start_col_index = df_raw.columns.get_loc('SiO2')
        end_col_col = 'Cs ' if 'Cs ' in df_raw.columns else 'Cs'
        if end_col_col not in df_raw.columns:
             end_col_index = df_raw.columns.get_loc('U')
        else:
             end_col_index = df_raw.columns.get_loc(end_col_col)
        cols_to_convert = df_raw.columns[start_col_index : end_col_index + 1]
    except KeyError: pass
    for col in cols_to_convert:
        if df_raw[col].dtype == 'object':
            df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

    # 2b. 定义 X (原始特征) 和 y (目标)
    cols_to_drop_initial = ['No.', 'Type', 'Type-1']
    cols_present = [col for col in cols_to_drop_initial if col in df_raw.columns]
    X_raw = df_raw.drop(columns=cols_present)
    y_raw = df_raw['Type']
    
    # 2c. 剔除高缺失列
    missing_percentage = X_raw.isnull().mean()
    cols_to_drop_missing = missing_percentage[missing_percentage > 0.4].index
    X_raw = X_raw.drop(columns=cols_to_drop_missing)
    # feature_names = X_raw.columns # 稍后在特征工程后更新
    
    # 2d. 编码和清理
    le = LabelEncoder()
    y_clean = y_raw.dropna()
    y_indices = y_clean.index
    X_clean = X_raw.loc[y_indices] # X_clean 是带 NaN 的 Pandas DataFrame
    y_encoded = le.fit_transform(y_clean) # y_encoded 是 NumPy 数组
    class_names_plot = list(le.classes_) # ['A-type', 'I-type', 'S-type']
    
    # --- 步骤 2.5/16: 【特征工程】 ---
    print("\n步骤 2.5/16: 正在执行特征工程 (创建 'ACNK_Boundary_Risk')...")
    boundary_value = 1.1
    distance = (X_clean['A/CNK'] - boundary_value).abs()
    distance = distance.replace(0, 1e-4) 
    X_clean['ACNK_Boundary_Risk'] = 1 / distance
    feature_names = X_clean.columns
    print(f"特征工程完毕。新特征 'ACNK_Boundary_Risk' 已添加。")
    
    # 2e. 拆分 (注意: X_train_raw 和 X_test_raw 保留了原始索引)
    X_train_raw, X_test_raw, y_train, y_test = train_test_split(
        X_clean, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )

    # --- 步骤 2.6/16: 【*** 全新步骤: 数据清洗 ***】 ---
    print("\n步骤 2.6/16: 正在执行数据清洗 (基于错误分析)...")
    
    # 我们需要一个 y_train 的 Pandas Series 来帮助我们做筛选
    y_train_series = pd.Series(y_train, index=X_train_raw.index)
    
    # 定义“边界”或“异常”样本的条件
    # 类别 2 (S-type) 且 A/CNK 接近 1.1 (例如 < 1.15)
    s_type_boundary_indices = X_train_raw[
        (y_train_series == 2) & (X_train_raw['A/CNK'] < 1.15)
    ].index
    
    # 类别 1 (I-type) 且 高场强元素异常高 (例如 > 300)
    i_type_anomaly_indices = X_train_raw[
        (y_train_series == 1) & (X_train_raw['Zr+Nb+Ce+Y'] > 300)
    ].index
    
    # 类别 0 (A-type) 且 高场强元素异常低 (例如 < 250)
    a_type_anomaly_indices = X_train_raw[
        (y_train_series == 0) & (X_train_raw['Zr+Nb+Ce+Y'] < 250)
    ].index
    
    # 合并所有要删除的索引
    indices_to_drop = s_type_boundary_indices.union(i_type_anomaly_indices).union(a_type_anomaly_indices)
    
    print(f"在训练集中找到 {len(s_type_boundary_indices)} 个 'S-type 边界' 样本。")
    print(f"在训练集中找到 {len(i_type_anomaly_indices)} 个 'I-type 异常' 样本。")
    print(f"在训练集中找到 {len(a_type_anomaly_indices)} 个 'A-type 异常' 样本。")
    print(f"总共将从训练集中移除 {len(indices_to_drop)} 个“模棱两可”的样本。")

    # 从训练集中删除这些样本
    X_train_raw_cleaned = X_train_raw.drop(indices_to_drop)
    y_train_cleaned = y_train_series.drop(indices_to_drop).values # .values 转回 NumPy 数组
    
    print(f"清洗后的训练集形状: {X_train_raw_cleaned.shape}")
    # --- 数据清洗结束 ---

    # 2f. 插补和缩放 (用于单次评估 和 *完整数据*)
    imputer = KNNImputer(n_neighbors=5)
    scaler = StandardScaler()
    
    # (用于单次评估)
    # 【*** 关键修改 ***】: 在 *清洗后* 的 X_train_raw_cleaned 上 fit
    X_train_processed = imputer.fit_transform(X_train_raw_cleaned) 
    X_train_processed = scaler.fit_transform(X_train_processed)
    
    # X_test 保持不变
    X_test_processed = imputer.transform(X_test_raw)
    X_test_processed = scaler.transform(X_test_processed)
    
    # (用于 CV, GridSearch, Validation Curve)
    # 【*** 关键修改 ***】: 交叉验证和调优现在也应该在 *清洗后* 的数据上进行
    # 我们从 X_clean 和 y_encoded 中移除这些“坏”样本
    X_all_raw_cleaned = X_clean.drop(indices_to_drop)
    y_all_cleaned = pd.Series(y_encoded, index=X_clean.index).drop(indices_to_drop).values
    
    # 在 *清洗后* 的完整数据上进行插补和缩放
    X_all_processed = imputer.fit_transform(X_all_raw_cleaned)
    X_all_processed = scaler.fit_transform(X_all_processed)
    y_all = y_all_cleaned # y_all 现在是 y_all_cleaned
    
    print("数据预处理完毕。")
    print(f"清洗后的训练集: {X_train_processed.shape}, 测试集: {X_test_processed.shape}, 清洗后的全部数据: {X_all_processed.shape}")
    # --- 预处理结束 ---

    # 3. SMOTE (用于单次训练)
    print("\n步骤 3/16: 正在应用 SMOTE 来平衡 *清洗后* 的训练数据...")
    smote = SMOTE(random_state=42, n_jobs=-1)
    # 【*** 关键修改 ***】: 使用 y_train_cleaned
    X_train_smote, y_train_smote = smote.fit_resample(X_train_processed, y_train_cleaned) 
    print(f"SMOTE 处理后训练集形状: {X_train_smote.shape}")

    # 4. 训练 XGBoost (单次)
    print("\n步骤 4/16: 正在使用 (SMOTE) 平衡后的数据训练 *单次* XGBoost 模型...")
    xgb_model_single = xgb.XGBClassifier(
        n_estimators=100, 
        random_state=42, 
        n_jobs=-1, 
        use_label_encoder=False, 
        eval_metric='mlogloss'
    )
    xgb_model_single.fit(X_train_smote, y_train_smote)
    print("模型训练完成。")

    # 5. 评估 XGBoost (单次)
    print("\n步骤 5/16: 正在使用 *原始* 测试集评估 (XGBoost) 模型...")
    y_pred_xgb = xgb_model_single.predict(X_test_processed)
    accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
    print(f"\n--- 模型评估结果 (SMOTE + XGBoost + 数据清洗) [单次 80/20 拆分] ---")
    print(f"\n[ 1. 总体准确率 ]")
    print(f"模型 (XGBoost) 准确率为: {accuracy_xgb * 100:.2f}%")
    
    # 6. 分类报告 (单次)
    print("\n[ 2. 详细分类报告 ]")
    target_names_report = ['A-type (Class 0)', 'I-type (Class 1)', 'S-type (Class 2)']
    report_xgb = classification_report(y_test, y_pred_xgb, target_names=target_names_report)
    print(report_xgb)

    # 7. 混淆矩阵 (单次)
    print("\n步骤 7/16: 正在生成混淆矩阵图表...")
    cm_xgb = confusion_matrix(y_test, y_pred_xgb)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='YlGnBu',
                xticklabels=class_names_plot, yticklabels=class_names_plot)
    plt.title(f'SMOTE + XGBoost (清洗后) 混淆矩阵\n(总体准确率: {accuracy_xgb * 100:.2f}%)')
    plt.ylabel('真实类别 (True Label)')
    plt.xlabel('预测类别 (Predicted Label)')
    plt.savefig("cm_xgboost_cleaned.png") # <-- 新文件名
    print(f"图表已保存为: cm_xgboost_cleaned.png")
    plt.close()

    # 8. 分类报告柱状图 (单次)
    print("\n步骤 8/16: 正在生成分类报告柱状图...")
    report_dict_xgb = classification_report(y_test, y_pred_xgb, target_names=class_names_plot, output_dict=True)
    report_df_xgb = pd.DataFrame(report_dict_xgb).transpose().loc[class_names_plot, ['precision', 'recall', 'f1-score']]
    report_df_xgb.plot(kind='bar', figsize=(12, 7), rot=0, colormap='summer')
    plt.title('SMOTE + XGBoost (清洗后) 分类报告指标')
    # ... (代码与上一版本相同)
    plt.savefig("report_bar_chart_xgb_cleaned.png") # <-- 新文件名
    print(f"图表已保存为: report_bar_chart_xgb_cleaned.png")
    plt.close()

    # 9. 特征重要性柱状图 (单次)
    print("\n步骤 9/16: 正在生成特征重要性柱状图 (XGBoost 模型)...")
    importances_xgb = xgb_model_single.feature_importances_
    feature_importance_xgb_df = pd.DataFrame({
        'Feature': feature_names, # feature_names 现在包含了新特征
        'Importance': importances_xgb
    }).sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(12, 9))
    sns.barplot(x='Importance', y='Feature', data=feature_importance_xgb_df.head(15), palette='autumn')
    plt.title('SMOTE + XGBoost (清洗后) - Top 15 重要特征')
    plt.xlabel('重要性得分 (F-Score / Gain)')
    plt.ylabel('特征 (Feature)')
    plt.savefig("feature_importance_xgb_cleaned.png") # <-- 新文件名
    print(f"图表已保存为: feature_importance_xgb_cleaned.png")
    plt.close()

    # 10. 特征散点图 (单次)
    print("\n步骤 10/16: 正在生成最重要特征散点图 (XGBoost 模型)...")
    top_feature_1_xgb = feature_importance_xgb_df.iloc[0]['Feature']
    top_feature_2_xgb = feature_importance_xgb_df.iloc[1]['Feature']
    # 注意: 这里我们使用 X_clean (包含 *所有* 样本, 甚至是我们删除的) 来绘图
    y_all_named_scatter = y_clean.map({0: 'A-type', 1: 'I-type', 2: 'S-type'})
    scatter_df_xgb = pd.DataFrame({'Feature_1': X_clean[top_feature_1_xgb], 'Feature_2': X_clean[top_feature_2_xgb], 'Type': y_all_named_scatter})
    plt.figure(figsize=(10, 7))
    sns.scatterplot(data=scatter_df_xgb, x='Feature_1', y='Feature_2', hue='Type', alpha=0.7, palette='dark')
    plt.title(f'XGBoost (清洗后) 模型 最重要特征散点图\n({top_feature_1_xgb} vs {top_feature_2_xgb})')
    plt.xlabel(top_feature_1_xgb)
    plt.ylabel(top_feature_2_xgb)
    plt.savefig("scatter_top_features_xgb_cleaned.png") # <-- 新文件名
    print(f"图表已保存为: scatter_top_features_xgb_cleaned.png")
    plt.close()

    # 11. 验证曲线 (折线图)
    print("\n步骤 11/16: 正在生成 XGBoost 验证曲线(折线图)... (这可能需要一点时间)")
    try:
        param_range = [1, 3, 5, 7, 10]
        # 【*** 关键修改 ***】: 在 *清洗后* 的 X_all_processed 和 y_all 上运行
        train_scores, test_scores = validation_curve(
            xgb.XGBClassifier(n_estimators=50, random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss'),
            X_all_processed, y_all, 
            param_name="max_depth",
            param_range=param_range,
            cv=3,
            scoring="accuracy",
            n_jobs=1 
        )
        # ... (绘图代码与上一版本相同)
        plt.title("XGBoost 验证曲线 (清洗后数据)")
        plt.savefig("validation_curve_xgb_cleaned.png") # <-- 新文件名
        print(f"图表已保存为: validation_curve_xgb_cleaned.png")
        plt.close()

    except Exception as e_val:
        print(f"验证曲线绘制失败: {e_val}")
    
    # --- 12. 10-折交叉验证 (XGBoost 默认参数) ---
    print("\n步骤 12/16: 正在执行 10-折交叉验证 (XGBoost 默认参数)... (这会需要几分钟时间)")
    
    # 流水线 *只包含* SMOTE 和 模型
    model_pipeline_xgb_default = ImbPipeline([
        ('smote', SMOTE(random_state=42, n_jobs=-1)),
        ('model', xgb.XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss'))
    ])
    
    k_folds = 10
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
    
    default_mean_accuracy_xgb = 0.0
    try:
        # 【*** 关键修改 ***】: 在 *清洗后* 的 X_all_processed 和 y_all 上运行
        cv_scores_xgb = cross_val_score(model_pipeline_xgb_default, X_all_processed, y_all, cv=skf, scoring='accuracy', n_jobs=1)
        
        print("\n--- 10-折交叉验证结果 (XGBoost 默认参数, 清洗后数据) ---")
        print(f"10 次试验的准确率分数: \n{np.round(cv_scores_xgb, 4)}")
        print("\n[ XGBoost (清洗后) 模型可信得分 ]")
        default_mean_accuracy_xgb = np.mean(cv_scores_xgb)
        print(f"平均准确率: {default_mean_accuracy_xgb * 100:.2f}%")
        print(f"标准差: +/- {np.std(cv_scores_xgb) * 100:.2f}%")
        print("(对比: 上一版 (带特征工程) 的得分是 91.95%)")
        
        print("\n正在生成交叉验证得分箱形图...")
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=cv_scores_xgb, orient='h', palette='Oranges')
        sns.stripplot(data=cv_scores_xgb, orient='h', color='black', alpha=0.7, jitter=0.1)
        plt.axvline(np.mean(cv_scores_xgb), color='blue', linestyle='--', label=f'平均值: {np.mean(cv_scores_xgb):.4f}')
        plt.title(f'10-折交叉验证得分分布 (SMOTE + 默认 XGBoost, 清洗后数据)')
        plt.xlabel('准确率 (Accuracy)')
        plt.xlim(0.8, 1.0)
        plt.legend()
        plt.savefig("cv_scores_boxplot_xgb_cleaned.png") # <-- 新文件名
        print(f"图表已保存为: cv_scores_boxplot_xgb_cleaned.png")
        plt.close()

    except Exception as e_cv:
        print(f"交叉验证执行失败: {e_cv}")
    
    # --- 13. 网格搜索 (Grid Search CV) for XGBoost ---
    print("\n步骤 13/16: 正在执行网格搜索 (Grid Search CV) 来寻找 XGBoost 最佳参数... (这会 *非常* 耗时, 可能需要 10-20 分钟)")

    param_grid_xgb = {
        'model__n_estimators': [100, 150],
        'model__max_depth': [5, 10, 15],
        'model__learning_rate': [0.1, 0.05]
    }
    
    # 流水线 *只包含* SMOTE 和 模型
    pipeline_for_gridsearch_xgb = ImbPipeline([
        ('smote', SMOTE(random_state=42, n_jobs=-1)),
        ('model', xgb.XGBClassifier(random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss'))
    ])

    grid_search_xgb = GridSearchCV(estimator=pipeline_for_gridsearch_xgb,
                                   param_grid=param_grid_xgb,
                                   cv=5, 
                                   scoring='accuracy',
                                   n_jobs=1,
                                   verbose=2)

    try:
        # 【*** 关键修改 ***】: 在 *清洗后* 的 X_all_processed 和 y_all 上运行
        grid_search_xgb.fit(X_all_processed, y_all)
        
        print("\n--- 网格搜索 (Grid Search) 结果 (XGBoost, 清洗后数据) ---")
        best_accuracy_xgb = grid_search_xgb.best_score_
        print(f"\n[ 最佳得分 ]")
        print(f"找到的最佳平均准确率 (来自 5-折 CV): {best_accuracy_xgb * 100:.2f}%")
        if default_mean_accuracy_xgb > 0:
            print(f"(对比: 默认 XGBoost 的 10-折 CV 平均准确率: {default_mean_accuracy_xgb * 100:.2f}%)")
        
        print("\n[ 最佳参数组合 ]")
        print("电脑找到的最佳参数设置是:")
        print(grid_search_xgb.best_params_)
        
        print("\n[ 性能提升 ]")
        if default_mean_accuracy_xgb > 0 and best_accuracy_xgb > default_mean_accuracy_xgb:
            print(f"通过调优, XGBoost 性能从 {default_mean_accuracy_xgb * 100:.2f}% 提升到了 {best_accuracy_xgb * 100:.2f}%。")
        elif default_mean_accuracy_xgb > 0:
            print(f"调优后的得分 {best_accuracy_xgb * 100:.2f}% 未能超越默认参数的 {default_mean_accuracy_xgb * 100:.2f}%。")
            print("这说明 XGBoost 的默认参数已经非常接近最优解了。")
        else:
            print("已找到最佳参数。")

    except Exception as e_grid:
        print(f"网格搜索执行失败: {e_grid}")
    
    # --- 14. 【*** 深入错误分析 ***】 ---
    print(f"\n步骤 14/16: 正在执行深入错误分析 (基于步骤 5 的 {accuracy_xgb*100:.2f}% 模型)...")
    
    try:
        # 1. 找到在 X_test_raw 中被错判的样本的 *原始索引*
        error_mask = y_pred_xgb != y_test
        # X_test_raw 是一个保留了原始索引的 Pandas DataFrame
        error_indices = X_test_raw[error_mask].index 
        
        print(f"模型在测试集上总共错判了 {len(error_indices)} 个样本。")
        
        # 2. 从 *原始* df_raw 中提取这些行
        misclassified_samples_raw = df_raw.loc[error_indices]
        
        # 3. 准备用于分析的 DataFrame
        predictions_for_errors = y_pred_xgb[error_mask]
        true_labels_for_errors = y_test[error_mask]
        
        # 4. 将 0,1,2 转换回 A,I,S
        true_labels_named = le.inverse_transform(true_labels_for_errors)
        pred_labels_named = le.inverse_transform(predictions_for_errors)
        
        analysis_df = misclassified_samples_raw.copy()
        analysis_df['True_Type'] = true_labels_named
        analysis_df['Predicted_Type'] = pred_labels_named
        
        # 5. 定义我们关心的地化指标 (加入我们的新特征!)
        key_geochem_cols = ['True_Type', 'Predicted_Type', 'A/CNK', 'ACNK_Boundary_Risk', 'Zr+Nb+Ce+Y', 'Zr', 'Nb', 'Y', 'Hf']
        # 确保这些列都存在
        existing_key_columns = [col for col in key_geochem_cols if col in analysis_df.columns]
        
        print("\n--- (清洗后模型) 所有错判样本的“关键”原始化学成分 ---")
        # .to_string() 保证打印所有行
        print(analysis_df[existing_key_columns].to_string())
        
        # 6. 执行您建议的具体分析
        print("\n--- (清洗后模型) 深入分析 1: 错判为 'I-type' 的 'S-type' (S-type 特征: A/CNK > 1.1) ---")
        s_as_i = analysis_df[
            (analysis_df['True_Type'] == 'S-type') & 
            (analysis_df['Predicted_Type'] == 'I-type')
        ]
        if not s_as_i.empty:
            print(s_as_i[existing_key_columns])
            print(f"分析: {len(s_as_i)} 个 S-type 被错判为 I-type。请检查它们的 'A/CNK' 值。")
            print(f"   它们的 A/CNK 均值: {s_as_i['A/CNK'].mean():.3f} (S-type 通常 > 1.1)")
        else:
            print("模型没有将 S-type 错判为 I-type。")

        print("\n--- (清洗后模型) 深入分析 2: 错判为 'A-type' 的 'I-type' (A-type 特征: Zr+Nb+Ce+Y 高) ---")
        i_as_a = analysis_df[
            (analysis_df['True_Type'] == 'I-type') & 
            (analysis_df['Predicted_Type'] == 'A-type')
        ]
        if not i_as_a.empty:
            print(i_as_a[existing_key_columns])
            print(f"分析: {len(i_as_a)} 个 I-type 被错判为 A-type。请检查它们的高场强元素。")
            print(f"   它们的 'Zr+Nb+Ce+Y' 均值: {i_as_a['Zr+Nb+Ce+Y'].mean():.2f}")
        else:
            print("模型没有将 I-type 错判为 A-type。")

        print("\n--- (清洗后模型) 深入分析 3: 被错判的 'A-type' ---")
        a_as_other = analysis_df[
            (analysis_df['True_Type'] == 'A-type')
        ]
        if not a_as_other.empty:
            print(a_as_other[existing_key_columns])
            print(f"分析: {len(a_as_other)} 个 A-type 被错判。请检查它们的高场强元素。")
            print(f"   它们的 'Zr+Nb+Ce+Y' 均值: {a_as_other['Zr+Nb+Ce+Y'].mean():.2f} (A-type 通常很高)")
        else:
            print("模型没有错判 A-type。")
            
    except Exception as e_error:
        print(f"错误分析执行失败: {e_error}")
    # --- 错误分析结束 ---

    # --- 步骤 15/16: 【*** 全新步骤: 原始模型 vs 清洗后模型 对比 ***】 ---
    print(f"\n步骤 15/16: 正在对比 '原始模型' vs '清洗后模型' 的可信得分...")
    
    # 1. 获取 "原始模型" (上一版) 的可信得分
    #    (我们从上一版日志中知道这个值是 91.95%)
    original_model_score = 91.95 # 这是上一版 (带特征工程) 的 10-fold CV 均值
    
    # 2. 获取 "清洗后模型" 的可信得分
    #    (来自本脚本的步骤 12)
    cleaned_model_score = default_mean_accuracy_xgb * 100
    
    print("\n--- 最终模型对比 ---")
    print(f"原始模型 (XGBoost + SMOTE + 特征工程): {original_model_score:.2f}%")
    print(f"清洗后模型 (XGBoost + SMOTE + 特征工程 + 数据清洗): {cleaned_model_score:.2f}%")
    
    if cleaned_model_score > original_model_score:
        print(f"\n[结论]: 成功! 通过清洗训练集中的“模糊”样本, 我们将模型的稳定准确率从 {original_model_score:.2f}% 提升到了 {cleaned_model_score:.2f}%。")
        print("这证明了那些“边界”样本正在“污染”训练过程。")
    else:
        print(f"\n[结论]: 有趣的发现! 清洗“模糊”样本反而使准确率从 {original_model_score:.2f}% 降至 {cleaned_model_score:.2f}%。")
        print("这说明模型 *需要* 看到那些“边界”样本来学习如何处理它们。")
        print(f"因此, {original_model_score:.2f}% 是我们能达到的最佳、最可信的分数。")
    # --- 对比结束 ---


    end_time_total = time.time()
    print(f"\n--- 完整流程结束 (总耗时: {(end_time_total - start_time_total) / 60:.2f} 分钟) ---")

except FileNotFoundError:
    print("\n--- 错误: 文件未找到 ---")
    print("找不到 'ASI分类.xls'。请确保它和您的 Jupyter 笔记本在同一个文件夹中。")
except ImportError:
    print(f"\n--- 错误: 缺少库 ---")
    print("请先安装 'xgboost', 'imbalanced-learn', 'matplotlib', 'seaborn', 'xlrd', 'openpyxl' 库。")
except Exception as e:
    print(f"\n--- 处理过程中发生错误 ---")
    print(f"错误详情: {e}")



中文字体 'SimHei' 加载成功。
--- 机器学习“王牌”模型 (XGBoost 完整分析) ---
步骤 1/16: 正在加载 *原始* Excel 文件 'ASI分类.xls'...
步骤 2/16: 正在清理和预处理数据...

步骤 2.5/16: 正在执行特征工程 (创建 'ACNK_Boundary_Risk')...
特征工程完毕。新特征 'ACNK_Boundary_Risk' 已添加。

步骤 2.6/16: 正在执行数据清洗 (基于错误分析)...
在训练集中找到 82 个 'S-type 边界' 样本。
在训练集中找到 66 个 'I-type 异常' 样本。
在训练集中找到 94 个 'A-type 异常' 样本。
总共将从训练集中移除 242 个“模棱两可”的样本。
清洗后的训练集形状: (830, 47)
数据预处理完毕。
清洗后的训练集: (830, 47), 测试集: (269, 47), 清洗后的全部数据: (1099, 47)

步骤 3/16: 正在应用 SMOTE 来平衡 *清洗后* 的训练数据...
SMOTE 处理后训练集形状: (1575, 47)

步骤 4/16: 正在使用 (SMOTE) 平衡后的数据训练 *单次* XGBoost 模型...
模型训练完成。

步骤 5/16: 正在使用 *原始* 测试集评估 (XGBoost) 模型...

--- 模型评估结果 (SMOTE + XGBoost + 数据清洗) [单次 80/20 拆分] ---

[ 1. 总体准确率 ]
模型 (XGBoost) 准确率为: 77.32%

[ 2. 详细分类报告 ]
                  precision    recall  f1-score   support

A-type (Class 0)       0.86      0.83      0.84       155
I-type (Class 1)       0.67      0.77      0.71        81
S-type (Class 2)       0.64      0.55      0.59        33

        accuracy                           0.77

E:\anaconda3\envs\rain_pytorch\lib\site-packages\ipykernel_launcher.py:244: UserWarning: Ignoring `palette` because no `hue` variable has been assigned.


图表已保存为: validation_curve_xgb_cleaned.png

步骤 12/16: 正在执行 10-折交叉验证 (XGBoost 默认参数)... (这会需要几分钟时间)

--- 10-折交叉验证结果 (XGBoost 默认参数, 清洗后数据) ---
10 次试验的准确率分数: 
[0.9545 0.9455 0.9636 0.9091 0.8818 0.9273 0.9545 0.9455 0.9545 0.945 ]

[ XGBoost (清洗后) 模型可信得分 ]
平均准确率: 93.81%
标准差: +/- 2.40%
(对比: 上一版 (带特征工程) 的得分是 91.95%)

正在生成交叉验证得分箱形图...
图表已保存为: cv_scores_boxplot_xgb_cleaned.png

步骤 13/16: 正在执行网格搜索 (Grid Search CV) 来寻找 XGBoost 最佳参数... (这会 *非常* 耗时, 可能需要 10-20 分钟)
Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV] END model__learning_rate=0.1, model__max_depth=5, model__n_estimators=100; total time=   0.3s
[CV] END model__learning_rate=0.1, model__max_depth=5, model__n_estimators=100; total time=   0.3s
[CV] END model__learning_rate=0.1, model__max_depth=5, model__n_estimators=100; total time=   0.3s
[CV] END model__learning_rate=0.1, model__max_depth=5, model__n_estimators=100; total time=   0.3s
[CV] END model__learning_rate=0.1, model__max_depth=5, model__n_estimators=100; total ti

In [7]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time

# --- 导入所有需要的模型和工具 ---
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC  # <-- 导入支持向量机
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import KNNImputer
# --- 导入结束 ---

# 忽略将来可能的警告
warnings.filterwarnings('ignore', category=FutureWarning)

# --- 设置 Matplotlib 支持中文 ---
try:
    plt.rcParams['font.sans-serif'] = ['SimHei']
    print("中文字体 'SimHei' 加载成功。")
except Exception as e:
    print(f"中文字体 'SimHei' 设置失败: {e}")
plt.rcParams['axes.unicode_minus'] = False
# --- 设置结束 ---


print("--- 机器学习“终极对决” (调优 RF vs 调优 SVM) ---")
start_time_total = time.time() # 记录总开始时间

try:
    # --- 步骤 1: 加载 *原始* Excel 文件并预处理 ---
    print("步骤 1/4: 正在加载并预处理数据 (包含特征工程和数据清洗)...")
    try:
        df_raw = pd.read_excel("ASI分类.xls", sheet_name="Sheet1", header=1)
    except FileNotFoundError:
        print("错误: 找不到 'ASI分类.xls'。")
        raise
    except ImportError:
        print("错误: 缺少 'xlrd' 库。请运行: !pip install xlrd")
        raise
        
    # 1a. 清理数字列
    cols_to_convert = []
    try:
        start_col_index = df_raw.columns.get_loc('SiO2')
        end_col_col = 'Cs ' if 'Cs ' in df_raw.columns else 'Cs'
        if end_col_col not in df_raw.columns:
             end_col_index = df_raw.columns.get_loc('U')
        else:
             end_col_index = df_raw.columns.get_loc(end_col_col)
        cols_to_convert = df_raw.columns[start_col_index : end_col_index + 1]
    except KeyError: pass
    for col in cols_to_convert:
        if df_raw[col].dtype == 'object':
            df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

    # 1b. 定义 X (原始特征) 和 y (目标)
    cols_to_drop_initial = ['No.', 'Type', 'Type-1']
    cols_present = [col for col in cols_to_drop_initial if col in df_raw.columns]
    X_raw = df_raw.drop(columns=cols_present)
    y_raw = df_raw['Type']
    
    # 1c. 剔除高缺失列
    missing_percentage = X_raw.isnull().mean()
    cols_to_drop_missing = missing_percentage[missing_percentage > 0.4].index
    X_raw = X_raw.drop(columns=cols_to_drop_missing)
    
    # 1d. 编码和清理
    le = LabelEncoder()
    y_clean = y_raw.dropna()
    y_indices = y_clean.index
    X_clean = X_raw.loc[y_indices]
    y_encoded = le.fit_transform(y_clean)
    
    # 1e. 特征工程
    boundary_value = 1.1
    distance = (X_clean['A/CNK'] - boundary_value).abs()
    distance = distance.replace(0, 1e-4) 
    X_clean['ACNK_Boundary_Risk'] = 1 / distance
    
    # 1f. 数据清洗 (关键步骤)
    # 拆分 *一次* 来决定要 drop 哪些 *训练* 样本
    X_train_raw_temp, _, y_train_temp, _ = train_test_split(
        X_clean, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
    )
    y_train_series_temp = pd.Series(y_train_temp, index=X_train_raw_temp.index)
    
    s_type_boundary_indices = X_train_raw_temp[
        (y_train_series_temp == 2) & (X_train_raw_temp['A/CNK'] < 1.15)
    ].index
    i_type_anomaly_indices = X_train_raw_temp[
        (y_train_series_temp == 1) & (X_train_raw_temp['Zr+Nb+Ce+Y'] > 300)
    ].index
    a_type_anomaly_indices = X_train_raw_temp[
        (y_train_series_temp == 0) & (X_train_raw_temp['Zr+Nb+Ce+Y'] < 250)
    ].index
    
    indices_to_drop = s_type_boundary_indices.union(i_type_anomaly_indices).union(a_type_anomaly_indices)
    
    # 从 *完整* 数据集 X_clean 和 y_encoded 中移除这些“坏”样本
    X_all_raw_cleaned = X_clean.drop(indices_to_drop)
    y_all_cleaned = pd.Series(y_encoded, index=X_clean.index).drop(indices_to_drop).values
    
    print(f"总共 {len(X_clean)} 个样本, 移除了 {len(indices_to_drop)} 个“模糊”样本。")
    print(f"最终用于交叉验证的数据集形状: {X_all_raw_cleaned.shape}")

    # 1g. 插补和缩放 (在清洗后的完整数据上)
    imputer = KNNImputer(n_neighbors=5)
    scaler = StandardScaler()
    
    X_all_processed = imputer.fit_transform(X_all_raw_cleaned)
    X_all_processed = scaler.fit_transform(X_all_processed)
    y_all = y_all_cleaned 
    
    print("数据预处理完毕。")
    # --- 预处理结束 ---

    # --- 步骤 2: 调优 随机森林 (Random Forest) ---
    print("\n步骤 2/4: 正在为 随机森林 执行网格搜索... (这会很耗时)")
    start_time_rf = time.time()
    
    # 创建 RF 流水线
    pipeline_rf = ImbPipeline([
        ('smote', SMOTE(random_state=42, n_jobs=-1)),
        ('model', RandomForestClassifier(random_state=42, n_jobs=-1))
    ])
    
    # 定义 RF 参数网格
    param_grid_rf = {
        'model__n_estimators': [100, 150, 200], # 树的数量
        'model__max_depth': [10, 20, None],     # 树的深度 (None 表示不限制)
        'model__min_samples_leaf': [1, 2]       # 叶节点最小样本数
    }
    # 总共 3 * 3 * 2 = 18 种组合
    
    grid_search_rf = GridSearchCV(estimator=pipeline_rf,
                                  param_grid=param_grid_rf,
                                  cv=5, 
                                  scoring='accuracy',
                                  n_jobs=1,
                                  verbose=1) # 打印进度
                                  
    try:
        grid_search_rf.fit(X_all_processed, y_all)
        best_score_rf = grid_search_rf.best_score_
        best_params_rf = grid_search_rf.best_params_
        print(f"随机森林 调优完成 (耗时: {(time.time() - start_time_rf) / 60:.2f} 分钟)")
    except Exception as e_rf:
        print(f"随机森林 调优失败: {e_rf}")
        best_score_rf = 0.0
        best_params_rf = {}

    # --- 步骤 3: 调优 支持向量机 (SVM) ---
    print("\n步骤 3/4: 正在为 支持向量机 (SVM) 执行网格搜索... (这会 *非常* 耗时)")
    start_time_svm = time.time()

    # 创建 SVM 流水线
    pipeline_svm = ImbPipeline([
        ('smote', SMOTE(random_state=42, n_jobs=-1)),
        ('model', SVC(random_state=42, probability=True)) # 开启 probability 以便未来分析
    ])
    
    # 定义 SVM 参数网格
    # SVM 对参数 C (惩罚系数) 和 gamma (核系数) 非常敏感
    param_grid_svm = {
        'model__C': [0.1, 1, 10],      # 惩罚系数
        'model__gamma': [0.1, 0.01, 'scale'], # 核系数
        'model__kernel': ['rbf']      # 只使用最高效的 RBF 核
    }
    # 总共 3 * 3 = 9 种组合
    
    grid_search_svm = GridSearchCV(estimator=pipeline_svm,
                                   param_grid=param_grid_svm,
                                   cv=5, 
                                   scoring='accuracy',
                                   n_jobs=1,
                                   verbose=1)
                                   
    try:
        grid_search_svm.fit(X_all_processed, y_all)
        best_score_svm = grid_search_svm.best_score_
        best_params_svm = grid_search_svm.best_params_
        print(f"支持向量机 调优完成 (耗时: {(time.time() - start_time_svm) / 60:.2f} 分钟)")
    except Exception as e_svm:
        print(f"支持向量机 调优失败: {e_svm}")
        best_score_svm = 0.0
        best_params_svm = {}
        
    # --- 步骤 4: 最终结果对比 ---
    print("\n步骤 4/4: 正在输出最终“王者对决”结果...")
    
    # 我们从上一版日志中知道 XGBoost 的“冠军”分数
    champion_score_xgb = 93.81 # (来自 XGBoost + 清洗后数据 + 10-折 CV)
    
    print("\n--- 最终模型对比 ---")
    print(f"卫冕冠军 (XGBoost + SMOTE + 清洗): {champion_score_xgb:.2f}%")
    print("---------------------------------")
    print(f"挑战者 1 (调优 RF + SMOTE + 清洗): {best_score_rf * 100:.2f}%")
    print(f"   (最佳参数: {best_params_rf})")
    print(f"挑战者 2 (调优 SVM + SMOTE + 清洗): {best_score_svm * 100:.2f}%")
    print(f"   (最佳参数: {best_params_svm})")
    
    final_winner_score = max(champion_score_xgb, best_score_rf * 100, best_score_svm * 100)
    
    print("\n--- 最终结论 ---")
    if final_winner_score == champion_score_xgb:
        print(f"XGBoost (93.81%) 守擂成功！它仍然是这个数据集上的最优模型。")
    elif final_winner_score == best_score_rf * 100:
        print(f"新的王者诞生了！ 调优后的随机森林 ({best_score_rf * 100:.2f}%) 成功超越了 XGBoost！")
    else:
        print(f"新的王者诞生了！ 调优后的支持向量机 (SVM) ({best_score_svm * 100:.2f}%) 成功超越了 XGBoost！")
        
    end_time_total = time.time()
    print(f"\n--- 完整流程结束 (总耗时: {(end_time_total - start_time_total) / 60:.2f} 分钟) ---")

except FileNotFoundError:
    print("\n--- 错误: 文件未找到 ---")
    print("找不到 'ASI分类.xls'。请确保它和您的 Jupyter 笔记本在同一个文件夹中。")
except ImportError:
    print(f"\n--- 错误: 缺少库 ---")
    print("请先安装 'xgboost', 'imbalanced-learn', 'matplotlib', 'seaborn', 'xlrd', 'openpyxl', 'scikit-learn' 库。")
except Exception as e:
    print(f"\n--- 处理过程中发生错误 ---")
    print(f"错误详情: {e}")


中文字体 'SimHei' 加载成功。
--- 机器学习“终极对决” (调优 RF vs 调优 SVM) ---
步骤 1/4: 正在加载并预处理数据 (包含特征工程和数据清洗)...
总共 1341 个样本, 移除了 242 个“模糊”样本。
最终用于交叉验证的数据集形状: (1099, 47)
数据预处理完毕。

步骤 2/4: 正在为 随机森林 执行网格搜索... (这会很耗时)
Fitting 5 folds for each of 18 candidates, totalling 90 fits
随机森林 调优完成 (耗时: 0.50 分钟)

步骤 3/4: 正在为 支持向量机 (SVM) 执行网格搜索... (这会 *非常* 耗时)
Fitting 5 folds for each of 9 candidates, totalling 45 fits
支持向量机 调优完成 (耗时: 0.21 分钟)

步骤 4/4: 正在输出最终“王者对决”结果...

--- 最终模型对比 ---
卫冕冠军 (XGBoost + SMOTE + 清洗): 93.81%
---------------------------------
挑战者 1 (调优 RF + SMOTE + 清洗): 90.00%
   (最佳参数: {'model__max_depth': 20, 'model__min_samples_leaf': 1, 'model__n_estimators': 100})
挑战者 2 (调优 SVM + SMOTE + 清洗): 87.72%
   (最佳参数: {'model__C': 1, 'model__gamma': 0.01, 'model__kernel': 'rbf'})

--- 最终结论 ---
XGBoost (93.81%) 守擂成功！它仍然是这个数据集上的最优模型。

--- 完整流程结束 (总耗时: 0.71 分钟) ---
